# Prediction ablation - tabular datasets

In [ ]:
import matplotlib
matplotlib.use('Agg')

import os
import sys
import pickle
import datetime
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, skew
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import auc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import tqdm as _tqdm_module
try:
    _tqdm_module.tqdm = lambda *a, **kw: (a[0] if a else iter([]))
    _tqdm_module.notebook.tqdm = lambda *a, **kw: (a[0] if a else iter([]))
    _tqdm_module.auto.tqdm = lambda *a, **kw: (a[0] if a else iter([]))
except AttributeError:
    pass


# configuration


DATASETS = ["wine", "california_housing", "adult", "diabetes"]

REPLOT_FROM_PKL = True # if model already trained
RUN_WILCOXON    = True


VALID_MASK_METHOD = "logit"   # "denom" or "logit", denom excludes bottom x percent of smallest denominators from normalization formula, logit by logit of predicted class. denom applicable to regression and classification, logit by classification only
DENOM_PERCENTILE  = 10      
LOGIT_IQR_FACTOR  = 0.5       #threshold = Q1 - FACTOR * IQR

SHAP_CONFIG = {
    "wine":               512,
    "california_housing": 512,
    "adult":              512,
    "diabetes":           512,
}

N_SAMPLES_CONFIG = {
    "wine":               100,
    "california_housing": 100,
    "adult":              100,
    "diabetes":           100,
}

N_RUNS         = 10
TEST_SIZE      = 0.2
RANDOM_STATE   = 6767
RBSHAP_SAMPLES = 25

YAXIS_PERCENTILE_LO = 2
YAXIS_PERCENTILE_HI = 98

OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA test"

MODELS    = ["random_forest", "xgboost", "mlp"]
BASELINES = ["mean", "median", "rbshap"]

MODEL_PARAMS = {
    "random_forest": {"n_estimators": 1000, "random_state": 423, "n_jobs": 8},
    "xgboost": {
        "n_estimators": 100, "random_state": 42, "max_depth": 5,
        "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8,
        "min_child_weight": 5, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "verbosity": 0, "n_jobs": 8
    },
    "mlp": {"hidden_size": 32, "epochs": 100, "lr": 0.001, "batch_size": 32}
}

np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)


# logit helper
def to_logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1.0 - p))


# MLP
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32), nn.ReLU(),
            nn.Linear(32, 32),         nn.ReLU(),
            nn.Linear(32, output_size)
        )
    def forward(self, x):
        return self.net(x)

class MLPWrapper:
    def __init__(self, params):
        self.params        = params
        self.model         = None
        self.is_classifier = False
        self.device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def fit(self, X_train, y_train):
        n_classes          = len(np.unique(y_train))
        self.is_classifier = (np.issubdtype(y_train.dtype, np.integer) and n_classes < 20)
        output_size        = n_classes if self.is_classifier else 1
        self.model         = MLP(X_train.shape[1], output_size).to(self.device)
        criterion          = nn.CrossEntropyLoss() if self.is_classifier else nn.MSELoss()
        y_tensor           = torch.LongTensor(y_train) if self.is_classifier else torch.FloatTensor(y_train)
        optimizer          = torch.optim.Adam(self.model.parameters(), lr=self.params['lr'])
        dataset            = TensorDataset(torch.FloatTensor(X_train), y_tensor)
        loader             = DataLoader(dataset, batch_size=self.params['batch_size'], shuffle=True)
        self.model.train()
        for _ in range(self.params['epochs']):
            for Xb, yb in loader:
                Xb = Xb.to(self.device); yb = yb.to(self.device)
                optimizer.zero_grad()
                pred = self.model(Xb)
                loss = criterion(pred, yb) if self.is_classifier else criterion(pred.squeeze(), yb)
                loss.backward(); optimizer.step()
        self.model.eval()

    def predict_raw(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()

    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.argmax(dim=1).cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()

class ModelWrapper:
    def __init__(self, model_type, params):
        self.model_type    = model_type
        self.params        = params
        self.model         = None
        self.is_classifier = False

    def fit(self, X_train, y_train):
        self.is_classifier = (np.issubdtype(y_train.dtype, np.integer) and len(np.unique(y_train)) < 20)
        if self.model_type == "random_forest":
            from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
            self.model = (RandomForestClassifier(**self.params) if self.is_classifier
                          else RandomForestRegressor(**self.params))
            self.model.fit(X_train, y_train)
        elif self.model_type == "xgboost":
            import xgboost as xgb
            self.model = (xgb.XGBClassifier(**self.params) if self.is_classifier
                          else xgb.XGBRegressor(**self.params))
            self.model.fit(X_train, y_train)
        elif self.model_type == "mlp":
            self.model = MLPWrapper(self.params)
            self.model.fit(X_train, y_train)

    def predict_scalar(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        if self.is_classifier:
            if self.model_type == "mlp":
                logits       = self.model.predict_raw(X)
                pred_classes = logits.argmax(axis=1)
                return logits[np.arange(len(logits)), pred_classes]
            else:
                proba        = self.model.predict_proba(X)
                pred_classes = proba.argmax(axis=1)
                p_k          = proba[np.arange(len(proba)), pred_classes]
                return to_logit(p_k)
        else:
            if self.model_type == "mlp":
                return self.model.predict_raw(X).flatten()
            return self.model.predict(X).flatten()

    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        if self.model_type == "mlp":
            return self.model.predict(X)
        return self.model.model.predict(X)

# load data

def load_data(dataset_name):
    if dataset_name == "california_housing":
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing()
        return d.data, d.target, list(d.feature_names)
    elif dataset_name == "diabetes":
        from sklearn.datasets import load_diabetes
        d = load_diabetes()
        return d.data, d.target, list(d.feature_names)
    elif dataset_name == "wine":
        from sklearn.datasets import load_wine
        d = load_wine()
        return d.data, d.target, list(d.feature_names)
    elif dataset_name == "adult":
        from sklearn.datasets import fetch_openml
        print("  Loading Adult dataset from OpenML...")
        data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
        X = data.data.copy(); y = data.target.copy()
        feature_names = X.columns.tolist()
        for col in X.columns:
            if X[col].dtype == 'object' or X[col].dtype.name == 'category':
                X[col] = LabelEncoder().fit_transform(X[col].astype(str))
        X = np.array(X.astype(np.float64))
        if y.dtype == 'object' or y.dtype.name == 'category':
            y = LabelEncoder().fit_transform(y.astype(str))
        return X, np.array(y), feature_names
    raise ValueError(f"Unknown dataset: {dataset_name}")

# baseline and ablation helpers

def compute_baseline(X_train, baseline_value):
    if baseline_value == "mean":
        return np.mean(X_train, axis=0)
    elif baseline_value == "median":
        return np.median(X_train, axis=0)
    elif baseline_value == "rbshap":
        idx = np.random.choice(len(X_train), size=RBSHAP_SAMPLES, replace=False)
        return X_train[idx]

def predict_ablated_scalar(model_wrapper, X_sample, features_to_remove, baseline):
    is_rbshap = isinstance(baseline, np.ndarray) and baseline.ndim == 2
    if len(features_to_remove) == 0:
        X_in = X_sample.reshape(1, -1)
    elif is_rbshap:
        X_in = np.tile(X_sample, (len(baseline), 1))
        X_in[:, list(features_to_remove)] = baseline[:, list(features_to_remove)]
    else:
        X_in = X_sample.copy()
        X_in[list(features_to_remove)] = baseline[list(features_to_remove)]
        X_in = X_in.reshape(1, -1)
    return float(np.mean(model_wrapper.predict_scalar(X_in)))

#compute valid mask

def compute_global_valid_mask(cached_orig, cached_empty, n_actual,
                               baselines, models, is_classifier,
                               method="denom"):
    min_denoms = np.full(n_actual, np.inf)
    for b in baselines:
        for m in models:
            d = np.abs(cached_orig[b][m] - cached_empty[b][m])
            min_denoms = np.minimum(min_denoms, d)

    # denom filter
    if method == "denom" or not is_classifier:
        if method == "logit" and not is_classifier:
            print("  logit filter not applicable for regression "
                  "using denom filter")
        threshold    = np.percentile(min_denoms, DENOM_PERCENTILE)
        global_valid = min_denoms >= threshold
        print(f"  denom threshold: (p{DENOM_PERCENTILE}): {threshold:.4e}  "
              f"| excluded: {int((~global_valid).sum())}")
        return global_valid, threshold, min_denoms

    # logit filter

    excluded_by = {i: set() for i in range(n_actual)}

    for m_name in models:
        ref_orig = cached_orig[baselines[0]][m_name]
        q1       = np.percentile(ref_orig, 25)
        q3       = np.percentile(ref_orig, 75)
        iqr      = q3 - q1
        thr      = q1 - LOGIT_IQR_FACTOR * iqr
        flagged  = np.where(ref_orig < thr)[0]
        print(f"  logit filter [{m_name:15s}]: "
              f"Q1={q1:.4f}, IQR={iqr:.4f}, "
              f"threshold={thr:.4f} (Q1 - {LOGIT_IQR_FACTOR}*IQR), "
              f"flagged={len(flagged)}")
        for idx in flagged:
            excluded_by[idx].add(m_name)

    global_valid = np.array([len(excluded_by[i]) == 0 for i in range(n_actual)])
    n_excl = int((~global_valid).sum())
    print(f"  logit filter total: {n_excl} samples excluded")
    for i in range(n_actual):
        if not global_valid[i]:
            print(f"    sample {i:3d} flagged by: {', '.join(sorted(excluded_by[i]))}")

    return global_valid, 0.0, min_denoms

# signed shap sorting

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse:
        order = order[::-1]
    return order

# aggregate curves

def aggregate_curves(curves, percentages, baselines, models):
    def agg(arr):
        a = np.array(arr)
        if len(a) == 0:
            return np.zeros_like(percentages), np.zeros_like(percentages)
        return a.mean(0), a.std(0) / np.sqrt(len(a))
    return {b: {m: {k: agg(curves[b][m][k]) for k in ("random", "topk", "bottomk")}
                for m in models} for b in baselines}

# y axis limits (variable)

def compute_ylims(agg_curves, baselines, models, plo, phi):
    all_vals = []
    for b in baselines:
        for m in models:
            for k in ("random", "topk", "bottomk"):
                all_vals.extend(agg_curves[b][m][k][0])
    return min(np.percentile(all_vals, plo), -0.05), max(np.percentile(all_vals, phi), 1.05)

#labels

MODEL_LABELS    = {"random_forest": "Random Forest", "xgboost": "XGBoost", "mlp": "MLP"}
BASELINE_LABELS = {"mean": "Mean", "median": "Median", "rbshap": f"RBShap (n={RBSHAP_SAMPLES})"}


#plot 3x3 grid plot
def plot_3x3(agg_c, abc_samples, percentages, DATASET, n_valid, n_actual,
             baselines, models, plot_fname, is_classifier):
    y_lo, y_hi  = compute_ylims(agg_c, baselines, models, YAXIS_PERCENTILE_LO, YAXIS_PERCENTILE_HI)
    score_label = "Logit of predicted class" if is_classifier else "Predicted value"
    filter_desc = (
        f"Logit filter: Q1 - {LOGIT_IQR_FACTOR}*IQR per model"
        if VALID_MASK_METHOD == "logit" and is_classifier
        else f"Denom filter: bottom {DENOM_PERCENTILE}th percentile excluded")

    fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharex=True, sharey=True)
    fig.suptitle(
        f"Prediction Ablation — {DATASET.replace('_', ' ').title()}  "
        f"(N={n_valid}/{n_actual} valid, seed={RANDOM_STATE})\n"
        f"Normalised {score_label}  |  "
        f"Ablation order: Top-K / Bottom-K by signed SHAP  |  "
        f"{filter_desc}",
        fontsize=11, fontweight='bold', y=1.02)

    for row, b_name in enumerate(baselines):
        for col, m_name in enumerate(models):
            ax = axes[row][col]
            d  = agg_c[b_name][m_name]
            rm, rs = d["random"]
            tm, ts = d["topk"]
            bm, bs = d["bottomk"]

            ax.fill_between(percentages, tm, bm, alpha=0.12, color='purple', label='Area between curves')
            ax.plot(percentages, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
            ax.fill_between(percentages, rm - rs, rm + rs, alpha=0.15, color='blue')
            ax.plot(percentages, tm, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
            ax.fill_between(percentages, tm - ts, tm + ts, alpha=0.15, color='red')
            ax.plot(percentages, bm, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')
            ax.fill_between(percentages, bm - bs, bm + bs, alpha=0.15, color='green')

            ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
            ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
            ax.set_xlim(0, 100); ax.set_ylim(y_lo, y_hi)
            ax.grid(True, alpha=0.3)

            n_s   = len(abc_samples[b_name][m_name])
            abc_m = np.mean(abc_samples[b_name][m_name]) if n_s > 0 else 0.0
            abc_s = np.std(abc_samples[b_name][m_name])  if n_s > 1 else 0.0
            ax.text(0.97, 0.97, f"ABC={abc_m:.3f}±{abc_s:.3f}\n(n={n_s})",
                    transform=ax.transAxes, fontsize=7, ha='right', va='top',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

            if row == 0: ax.set_title(MODEL_LABELS[m_name], fontsize=11, fontweight='bold')
            if col == 0:
                ax.set_ylabel("Prediction as fraction of original", fontsize=8)
                ax.text(-0.28, 0.5, BASELINE_LABELS[b_name], transform=ax.transAxes,
                        fontsize=11, fontweight='bold', va='center', ha='center', rotation=90)
            if col == 2: ax.tick_params(axis='y', labelright=True)
            if row == 2: ax.set_xlabel("Ablated features (%)", fontsize=9)

    handles, labels_leg = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels_leg, loc='lower center', ncol=4, fontsize=11, bbox_to_anchor=(0.5, -0.03))
    plt.tight_layout()
    plt.savefig(plot_fname, bbox_inches='tight')
    print(f"  Plot saved -> {plot_fname}")
    plt.show()

# wilcoxon signed rank test

def run_wilcoxon(abc_samples, DATASET, models, diag_fname):
    pairs       = [("mean", "median"), ("mean", "rbshap"), ("median", "rbshap")]
    pair_labels = ["Mean vs Median", "Mean vs RBShap", "Median vs RBShap"]
    alpha       = 0.05

    print(f" symmetry check and wilcoxon test — {DATASET.upper()}")

    sym_results = {m: {p: None for p in pair_labels} for m in models}

    for m_name in models:
        print(f"  {MODEL_LABELS[m_name]}")
        for (b1, b2), plabel in zip(pairs, pair_labels):
            v1 = np.array(abc_samples[b1][m_name])
            v2 = np.array(abc_samples[b2][m_name])
            n  = min(len(v1), len(v2))
            v1, v2 = v1[:n], v2[:n]
            diff = v1 - v2
            if len(diff) < 3:
                print(f"    {plabel}: too few values"); continue
            s   = float(skew(diff))
            rel = abs(np.mean(diff) - np.median(diff)) / (np.std(diff) + 1e-10)
            print(f"    {plabel}: skew={s:+.3f}, |mu-med|/sigma={rel:.3f}")
            sym_results[m_name][plabel] = (s, rel)
        print()

    fig2, axes2 = plt.subplots(len(models), len(pairs),
                                figsize=(5*len(pairs), 4*len(models)), squeeze=False)
    fig2.suptitle(f"Symmetry Diagnostics — {DATASET.replace('_', ' ').title()}\n"
                  f"Differences of ABC_pred between baselines",
                  fontsize=12, fontweight='bold')

    for row, m_name in enumerate(models):
        for col, ((b1, b2), plabel) in enumerate(zip(pairs, pair_labels)):
            ax   = axes2[row][col]
            v1   = np.array(abc_samples[b1][m_name])
            v2   = np.array(abc_samples[b2][m_name])
            n    = min(len(v1), len(v2))
            v1, v2 = v1[:n], v2[:n]
            diff = v1 - v2
            ax.hist(diff, bins=max(5, n // 5), color='steelblue', edgecolor='white', alpha=0.85)
            ax.axvline(0,               color='red',    lw=1.5, ls='--', label='0')
            ax.axvline(np.mean(diff),   color='orange', lw=1.5, ls='-',  label=f'mean={np.mean(diff):.3f}')
            ax.axvline(np.median(diff), color='green',  lw=1.5, ls=':',  label=f'median={np.median(diff):.3f}')
            res  = sym_results[m_name][plabel]
            info = f"skew={res[0]:+.2f}, |mu-med|/sigma={res[1]:.2f}" if res else ""
            ax.set_title(f"{MODEL_LABELS[m_name]} | {plabel}\n{info}", fontsize=8)
            ax.set_xlabel(f"ABC_pred difference ({b1} - {b2})", fontsize=8)
            ax.set_ylabel("Count", fontsize=8)
            ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(diag_fname, bbox_inches='tight')
    print(f"  diagnostics saved -> {diag_fname}\n")
    plt.show()

    print(f"  WILCOXON SIGNED-RANK TEST  (alpha = {alpha})\n")
    for m_name in models:
        print(f"  {MODEL_LABELS[m_name]}")
        for (b1, b2), plabel in zip(pairs, pair_labels):
            v1 = np.array(abc_samples[b1][m_name])
            v2 = np.array(abc_samples[b2][m_name])
            n  = min(len(v1), len(v2))
            v1, v2 = v1[:n], v2[:n]
            diff = v1 - v2
            if np.all(diff == 0) or len(diff) < 3:
                print(f"    {plabel}: skipped"); continue
            try:
                stat, p = wilcoxon(v1, v2, alternative='two-sided')
                sig    = "significant" if p < alpha else "not significant"
                better = b1 if np.mean(v1) > np.mean(v2) else b2
                print(f"    {plabel}: W={stat:.1f}, p={p:.4f} -> {sig} | "
                      f"higher: {better} | "
                      f"Dmean={np.mean(diff):+.4f}, Dmedian={np.median(diff):+.4f}")
            except Exception as e:
                print(f"    {plabel}: error - {str(e)[:60]}")
        print()

# replot from pkl (if model is trained and you want to change the output plot for example)

def replot_from_pkl(DATASET):
    pkl_path = f"{OUTPUT_DIR}/prediction_ablation_{DATASET}.pkl"
    print(f"\n  loading {pkl_path} ...")
    with open(pkl_path, "rb") as f:
        saved = pickle.load(f)

    percentages   = np.array(saved["percentages"])
    n_actual      = saved["meta"]["n_samples"]
    is_classifier = saved["meta"]["is_classifier"]
    pss           = saved["per_sample_store"]

    global_valid = np.array(saved["global_valid"])
    n_valid      = int(global_valid.sum())
    print(f"  global valid samples: {n_valid}/{n_actual} ({n_actual - n_valid} excluded)\n")

    abc_samples = {b: {m: [] for m in MODELS} for b in BASELINES}

    #load per-sample ABC values
    for b in BASELINES:
        for m in MODELS:
            for entry in pss[b][m]:
                if not global_valid[entry["index"]]:
                    continue
                abc_samples[b][m].append(entry["abc"])

    #load aggragated curves
    agg_c = {b: {m: {} for m in MODELS} for b in BASELINES}
    for b in BASELINES:
        for m in MODELS:
            saved_agg = saved["agg_curves"][b][m]
            agg_c[b][m] = {
                "random":  (np.array(saved_agg["random"]["mean"]),
                            np.array(saved_agg["random"]["se"])),
                "topk":    (np.array(saved_agg["topk"]["mean"]),
                            np.array(saved_agg["topk"]["se"])),
                "bottomk": (np.array(saved_agg["bottomk"]["mean"]),
                            np.array(saved_agg["bottomk"]["se"])),
            }

    plot_fname = f"{OUTPUT_DIR}/prediction_ablation_{DATASET}.pdf"
    plot_3x3(agg_c, abc_samples, percentages, DATASET, n_valid, n_actual,
             BASELINES, MODELS, plot_fname, is_classifier)

    if RUN_WILCOXON:
        diag_fname = f"{OUTPUT_DIR}/prediction_wilcoxon_diagnostics_{DATASET}.pdf"
        run_wilcoxon(abc_samples, DATASET, MODELS, diag_fname)

# main loop

import shap as shap_lib
import joblib
from scipy.linalg import lstsq
from math import comb


# own kernelshap, identical to kernelshap but coalitions fixed across
#all baseline model combinations for the same sample

def shapley_kernel_weight(z, d):
    s = int(z.sum())
    if s == 0 or s == d:
        return 1e6
    return (d - 1) / (comb(d, s) * s * (d - s))

def sample_coalitions(n_feat, n_samples, seed):
    rng = np.random.RandomState(seed)
    Z   = rng.randint(0, 2, size=(n_samples, n_feat)).astype(float)
    Z[0] = np.zeros(n_feat)
    Z[1] = np.ones(n_feat)
    return Z

def kernel_shap_wls(Z, W, model_fn, phi_0):
    y      = model_fn(Z) - phi_0
    W_sqrt = np.sqrt(W)
    ZW     = Z * W_sqrt[:, None]
    yW     = y * W_sqrt
    phi, _, _, _ = lstsq(ZW, yW)
    return phi

def compute_shap_fixed(Z, W, mw, X_sample, baseline, empty_scalar):
    def model_fn(presence_matrix):
        preds = []
        for row in presence_matrix:
            absent = list(np.where(row == 0)[0])
            preds.append(predict_ablated_scalar(mw, X_sample, absent, baseline))
        return np.array(preds)
    return kernel_shap_wls(Z, W, model_fn, empty_scalar)

for DATASET in DATASETS:
    print(f"  DATASET: {DATASET.upper()}")

    pkl_path = f"{OUTPUT_DIR}/prediction_ablation_{DATASET}.pkl"
    if REPLOT_FROM_PKL and os.path.exists(pkl_path):
        print(f"replotting from pkl")
        replot_from_pkl(DATASET)
        continue

    X, y, feature_names = load_data(DATASET)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    n_actual   = min(N_SAMPLES_CONFIG[DATASET], len(X_test_sc))
    X_analysis = X_test_sc[:n_actual]

    is_classifier = (np.issubdtype(y.dtype, np.integer) and len(np.unique(y)) < 20)
    n_feat        = X_analysis.shape[1]
    percentages   = np.linspace(0, 100, n_feat + 1)
    shap_mode     = SHAP_CONFIG[DATASET]

    print(f"  Task:     {'Classification' if is_classifier else 'Regression'}")
    print(f"  Features: {n_feat}  |  Samples: {n_actual}  |  SHAP coalitions: {shap_mode}")
    print(f"  Filter:   {VALID_MASK_METHOD}"
          + (f" (IQR factor: {LOGIT_IQR_FACTOR})" if VALID_MASK_METHOD == "logit"
             else f" (percentile: {DENOM_PERCENTILE})") + "\n")

    print("training....")
    trained_models = {}
    for m_name in MODELS:
        print(f"    {m_name}...", flush=True)
        w = ModelWrapper(m_name, MODEL_PARAMS[m_name])
        w.fit(X_train_sc, y_train)
        trained_models[m_name] = w
        if m_name == "mlp":
            torch.save(w.model.model.state_dict(), f"{OUTPUT_DIR}/model_{DATASET}_{m_name}.pt")
        else:
            joblib.dump(w, f"{OUTPUT_DIR}/model_{DATASET}_{m_name}.pkl")
    print(f"  models saved\n")

    all_baselines = {b: compute_baseline(X_train_sc, b) for b in BASELINES}

    cached_orig  = {b: {m: None for m in MODELS} for b in BASELINES}
    cached_empty = {b: {m: None for m in MODELS} for b in BASELINES}
    for b_name_tmp, baseline_tmp in all_baselines.items():
        for m_name_tmp in MODELS:
            mw_tmp = trained_models[m_name_tmp]
            cached_orig[b_name_tmp][m_name_tmp] = np.array([
                predict_ablated_scalar(mw_tmp, x, [], baseline_tmp) for x in X_analysis])
            cached_empty[b_name_tmp][m_name_tmp] = np.array([
                predict_ablated_scalar(mw_tmp, x, list(range(n_feat)), baseline_tmp)
                for x in X_analysis])

    global_valid, denom_threshold, min_denoms = compute_global_valid_mask(
        cached_orig, cached_empty, n_actual, BASELINES, MODELS,
        is_classifier, method=VALID_MASK_METHOD)
    n_valid = global_valid.sum()
    print(f"  global valid samples:{n_valid}/{n_actual} ({n_actual - n_valid} excluded)\n")

    curves           = {b: {m: {"random": [], "topk": [], "bottomk": []} for m in MODELS} for b in BASELINES}
    abc_samples      = {b: {m: [] for m in MODELS} for b in BASELINES}
    per_sample_store = {b: {m: [] for m in MODELS} for b in BASELINES}

    # random ablation, permutations generated once per run, identical across
    # all baseline-model combinations
    np.random.seed(RANDOM_STATE + 200000)
    fixed_perms = [np.random.permutation(n_feat) for _ in range(N_RUNS)]

    for b_name in BASELINES:
        baseline = all_baselines[b_name]
        for m_name in MODELS:
            mw            = trained_models[m_name]
            orig_scalars  = cached_orig[b_name][m_name]
            empty_scalars = cached_empty[b_name][m_name]
            denoms        = orig_scalars - empty_scalars

            for run_idx in range(N_RUNS):
                perm  = fixed_perms[run_idx]
                curve = []
                for k in range(n_feat + 1):
                    vals = []
                    for j, x in enumerate(X_analysis):
                        if not global_valid[j]: continue
                        raw = predict_ablated_scalar(mw, x, list(perm[:k]), baseline)
                        vals.append((raw - empty_scalars[j]) / denoms[j])
                    curve.append(float(np.mean(vals)) if vals else 0.0)
                curves[b_name][m_name]["random"].append(
                    np.interp(percentages, np.linspace(0, 100, len(curve)), curve))

    # coalitions generated once and then reused   
    print(f"  computing shap values with fixed coalitions...")
    for i, X_sample in enumerate(X_analysis):
        if not global_valid[i]: continue

        Z = sample_coalitions(n_feat, shap_mode, seed=RANDOM_STATE + i)
        W = np.array([shapley_kernel_weight(z, n_feat) for z in Z])

        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{n_actual} samples", flush=True)

        for b_name in BASELINES:
            baseline = all_baselines[b_name]
            for m_name in MODELS:
                mw           = trained_models[m_name]
                orig_scalar  = cached_orig[b_name][m_name][i]
                empty_scalar = cached_empty[b_name][m_name][i]
                denom        = orig_scalar - empty_scalar

                try:
                    sv = compute_shap_fixed(Z, W, mw, X_sample,
                                            baseline, empty_scalar)

                    order_topk    = get_topk_order(sv, orig_scalar, empty_scalar, inverse=False)
                    order_bottomk = get_topk_order(sv, orig_scalar, empty_scalar, inverse=True)

                    topk_curve, bottomk_curve = [], []
                    for k in range(n_feat + 1):
                        raw_tk = predict_ablated_scalar(mw, X_sample, list(order_topk[:k]),    baseline)
                        raw_bk = predict_ablated_scalar(mw, X_sample, list(order_bottomk[:k]), baseline)
                        topk_curve.append((raw_tk - empty_scalar) / denom)
                        bottomk_curve.append((raw_bk - empty_scalar) / denom)

                    pct_real       = np.linspace(0, 100, n_feat + 1)
                    topk_interp    = np.interp(percentages, pct_real, topk_curve)
                    bottomk_interp = np.interp(percentages, pct_real, bottomk_curve)

                    curves[b_name][m_name]["topk"].append(topk_interp)
                    curves[b_name][m_name]["bottomk"].append(bottomk_interp)

                    abc_s = float(auc(percentages / 100, bottomk_interp - topk_interp))
                    abc_samples[b_name][m_name].append(abc_s)

                    per_sample_store[b_name][m_name].append({
                        "index":        i,
                        "orig_scalar":  orig_scalar,
                        "empty_scalar": empty_scalar,
                        "shap_values":  sv.tolist(),
                        "topk_norm":    topk_curve,
                        "bottomk_norm": bottomk_curve,
                        "abc":          abc_s,
                    })

                except Exception as e:
                    print(f"  SHAP error sample {i} {b_name} {m_name}: {str(e)[:60]}")
                    continue

    for b_name in BASELINES:
        for m_name in MODELS:
            n_abc = len(abc_samples[b_name][m_name])
            print(f"  {b_name.upper()} | {m_name.upper()}: "
                  f"ABC_pred = {np.mean(abc_samples[b_name][m_name]):.4f} "
                  f"+/- {np.std(abc_samples[b_name][m_name]):.4f}  (n={n_abc})")

    agg_c = aggregate_curves(curves, percentages, BASELINES, MODELS)

    with open(pkl_path, "wb") as f:
        pickle.dump({
            "meta": {
                "dataset":           DATASET,
                "is_classifier":     is_classifier,
                "n_samples":         n_actual,
                "n_valid":           int(n_valid),
                "valid_mask_method": VALID_MASK_METHOD,
                "denom_percentile":  DENOM_PERCENTILE,
                "denom_threshold":   float(denom_threshold),
                "logit_iqr_factor":  LOGIT_IQR_FACTOR,
                "shap_coalitions":   shap_mode,
                "n_runs":            N_RUNS,
                "random_state":      RANDOM_STATE,
                "saved_at":          datetime.datetime.now().isoformat(),
            },
            "percentages":      percentages.tolist(),
            "global_valid":     global_valid.tolist(),
            "min_denoms":       min_denoms.tolist(),
            "abc_samples":      abc_samples,
            "per_sample_store": per_sample_store,
            "agg_curves": {
                b: {m: {k: {"mean": v[0].tolist(), "se": v[1].tolist()}
                        for k, v in agg_c[b][m].items()}
                    for m in MODELS}
                for b in BASELINES
            },
        }, f)
    print(f"  results saved: {pkl_path}")

    plot_fname = f"{OUTPUT_DIR}/prediction_ablation_{DATASET}.pdf"
    plot_3x3(agg_c, abc_samples, percentages, DATASET, n_valid, n_actual,
             BASELINES, MODELS, plot_fname, is_classifier)

    if RUN_WILCOXON:
        diag_fname = f"{OUTPUT_DIR}/prediction_wilcoxon_diagnostics_{DATASET}.pdf"
        run_wilcoxon(abc_samples, DATASET, MODELS, diag_fname)

print(f" all outputs in {OUTPUT_DIR}/")


# Prediction ablation - DistilBERT

In [ ]:
%matplotlib inline
import subprocess
import sys

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])

try:
    from datasets import load_dataset
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets"])

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, skew
from scipy.linalg import lstsq
from sklearn.metrics import auc
from math import comb
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import pickle, datetime

# configuration

RANDOM_SEED = 67676767
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

N_SAMPLES = 30
N_RUNS    = 20
SHAP_MODE = 512

VALID_MASK_METHOD = "logit"
DENOM_PERCENTILE  = 10
LOGIT_IQR_FACTOR  = 0.5

BASELINES = {
    "mask":    "[MASK]",
    "removal": None,
}

SAVE_PATH  = "distilbert_sst2_results.pkl"
PLOT_PATH  = r"C:\Users\lukas\OneDrive\Documents\Plots MA\distilbert_ablation_analysis.pdf"
DIAG_PATH  = r"C:\Users\lukas\OneDrive\Documents\Plots MA\distilbert_wilcoxon_diagnostics.pdf"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Random seed: {RANDOM_SEED}")
print(f"SHAP mode:   {SHAP_MODE}")
print(f"Filter:      {VALID_MASK_METHOD}"
      + (f" (IQR factor: {LOGIT_IQR_FACTOR})" if VALID_MASK_METHOD == "logit"
         else f" (percentile: {DENOM_PERCENTILE})") + "\n")

#load model

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"
print("Loading DistilBERT model...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.\n")

# prediction helpers

def predict_logits(texts):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(texts, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits.cpu().numpy()

def ablate_text(words, indices_to_remove, baseline_type):
    if baseline_type == "mask":
        return " ".join(
            "[MASK]" if i in indices_to_remove else w
            for i, w in enumerate(words))
    else:
        kept = [w for i, w in enumerate(words) if i not in indices_to_remove]
        return " ".join(kept) if kept else "[PAD]"

def get_empty_logit(words, predicted_class, baseline_type):
    empty_text = ablate_text(words, set(range(len(words))), baseline_type)
    return float(predict_logits(empty_text)[0, predicted_class])

# own kernelshap (fixed coalitions)

def shapley_kernel_weight(z, d):
    s = int(z.sum())
    if s == 0 or s == d:
        return 1e6
    return (d - 1) / (comb(d, s) * s * (d - s))

def sample_coalitions(n_feat, n_samples, seed):
    rng = np.random.RandomState(seed)
    Z   = rng.randint(0, 2, size=(n_samples, n_feat)).astype(float)
    Z[0] = np.zeros(n_feat)
    Z[1] = np.ones(n_feat)
    return Z

def kernel_shap_wls(Z, W, model_fn, phi_0):
    y      = model_fn(Z) - phi_0
    W_sqrt = np.sqrt(W)
    ZW     = Z * W_sqrt[:, None]
    yW     = y * W_sqrt
    phi, _, _, _ = lstsq(ZW, yW)
    return phi

def compute_shap_fixed(Z, W, words, predicted_class, baseline_type, empty_logit):
    def model_fn(Z_in):
        preds = []
        for row in Z_in:
            absent = {i for i, p in enumerate(row) if p == 0}
            text   = ablate_text(words, absent, baseline_type)
            preds.append(float(predict_logits(text)[0, predicted_class]))
        return np.array(preds)
    return kernel_shap_wls(Z, W, model_fn, empty_logit)

# valid mask

def compute_global_valid_mask(cached_orig, cached_empty, n_actual,
                               baselines, all_pred, method="denom"):
    min_denoms = np.full(n_actual, np.inf)
    for b in baselines:
        d = np.abs(cached_orig[b] - cached_empty[b])
        min_denoms = np.minimum(min_denoms, d)

    if method == "denom":
        threshold    = np.percentile(min_denoms, DENOM_PERCENTILE)
        global_valid = min_denoms >= threshold
        print(f"  denom threshold (p{DENOM_PERCENTILE}): {threshold:.4e}  "
              f"| excluded: {int((~global_valid).sum())}")
        return global_valid, threshold, min_denoms

    ref_orig     = cached_orig[baselines[0]]
    q1           = np.percentile(ref_orig, 25)
    q3           = np.percentile(ref_orig, 75)
    iqr          = q3 - q1
    thr          = q1 - LOGIT_IQR_FACTOR * iqr
    flagged      = np.where(ref_orig < thr)[0]
    global_valid = np.ones(n_actual, dtype=bool)
    print(f"  Logit filter: Q1={q1:.4f}, IQR={iqr:.4f}, "
          f"threshold={thr:.4f} (Q1 - {LOGIT_IQR_FACTOR}*IQR), "
          f"flagged={len(flagged)}")
    for idx in flagged:
        global_valid[idx] = False

    n_excl = int((~global_valid).sum())
    print(f"  Logit filter total: {n_excl} sample(s) excluded")
    for i in range(n_actual):
        if not global_valid[i]:
            cls_label = "pos" if all_pred[i] == 1 else "neg"
            print(f"    sample {i:3d} [{cls_label}] logit={ref_orig[i]:.4f}  "
                  f"{SENTIMENT_TEXTS[i][:60]}")
    return global_valid, 0.0, min_denoms

# sentences

SENTIMENT_TEXTS = [
    "I remember when this came out a lot of kids were nuts about it",
    "This is a great movie for the true romantics and sports lovers alike",
    "This is strictly a review of the pilot episode as it appears on DVD",
    "I was quite impressed with this movie as a child of eight or nine",
    "When Family Guy first premiered I was not in a discriminating mood",
    "Tom Clancy uses Alesandr Nevsky in his book Red Storm Rising",
    "I won't spend a lot of time nor energy on this comment",
    "This was such a terrible film almost a comedy sketch of a noir film",
    "Well maybe I'm just having a bad run with Hindi movies lately",
    "The Duke is a very silly film--a dog becoming a duke",
    "More exciting than the Wesley Snipes film and with better characters too",
    "This movie was Jerry Bruckheimer's idea to sell some records",
    "Karl Jr and his dad are now running an army on a remote island",
    "The film has no connection with the real life in Bosnia in those days",
    "No doubt Frank Sinatra was a talented actor as well as a talented singer",
    "Yesterday my Spanish Catalan wife and myself saw this emotional lesson in history",
    "The 3rd and in my view the best of the Blackadder series",
    "With some films it is really hard to tell for whom they were made",
    "Not only is this film entertaining with excellent comedic acting but also interesting politically",
    "This was Eisenstein's first completed project in over ten years",
    "Many of these other viewers complain that the story line has already been attempted",
    "Oh boy it's another comet-hitting-the-earth film",
    "First of all DO NOT call this a remake of the 63 film",
    "Great little short film that aired a while ago on SBS here in Aus",
    "If you really loved GWTW you will find quite disappointing the story",
    "Probably New Zealands worst Movie ever made The Jokes They are not funny",
    "This film is about the worst I have seen in a very long time",
    "This is by far the worst movie I have ever seen in the cinema",
    "If anybody really wants to understand Hitler read WWI history not WWII history",
    "This movie is more Lupin then most especially coming from Funimation",
][:N_SAMPLES]

# pass 1

print(f"  disitlbert ablation analysis")

MAX_WORDS = max(len(t.split()) for t in SENTIMENT_TEXTS)
N_GRID    = MAX_WORDS + 1
PCT_GRID  = np.linspace(0, 100, N_GRID)
print(f"  longest sentence: {MAX_WORDS} words, {N_GRID} grid points\n")

print("Pass 1: collecting orig/empty scalars")
cached_orig  = {b: np.full(N_SAMPLES, np.nan) for b in BASELINES}
cached_empty = {b: np.full(N_SAMPLES, np.nan) for b in BASELINES}
all_words    = []
all_pred     = []

for idx, text in enumerate(SENTIMENT_TEXTS):
    words           = text.split()
    logits_orig     = predict_logits(text)[0]
    predicted_class = int(np.argmax(logits_orig))
    orig_logit      = float(logits_orig[predicted_class])
    all_words.append(words)
    all_pred.append(predicted_class)
    for b_name in BASELINES:
        cached_orig[b_name][idx]  = orig_logit
        cached_empty[b_name][idx] = get_empty_logit(words, predicted_class, b_name)

#global valid mask

print(f"\nComputing valid mask (method={VALID_MASK_METHOD})...")
global_valid, denom_threshold, min_denoms = compute_global_valid_mask(
    cached_orig, cached_empty, N_SAMPLES, list(BASELINES.keys()),
    all_pred, method=VALID_MASK_METHOD)
n_valid = int(global_valid.sum())
print(f"Global valid samples: {n_valid}/{N_SAMPLES} "
      f"({N_SAMPLES - n_valid} excluded)\n")

# Pass 2: sample outer loop to fix coalitions for SHAP. Ranodm ablation permutations also fixed

curves          = {b: {"random": [], "topk": [], "bottomk": []} for b in BASELINES}
abc_per_sample  = {b: [] for b in BASELINES}
per_sample_data = {b: [] for b in BASELINES}

print("Pass 2: computing SHAP + ablation curves...\n")

for idx in range(N_SAMPLES):
    if not global_valid[idx]:
        continue

    words           = all_words[idx]
    predicted_class = all_pred[idx]
    n_words         = len(words)
    pct_real        = np.linspace(0, 100, n_words + 1)

    # Generate coalitions once per sample
    Z = sample_coalitions(n_words, SHAP_MODE, seed=RANDOM_SEED + idx)
    W = np.array([shapley_kernel_weight(z, n_words) for z in Z])

    if (idx + 1) % 10 == 0:
        print(f"  {idx+1}/{N_SAMPLES} samples processed", flush=True)

    for b_name in BASELINES:
        orig_logit  = cached_orig[b_name][idx]
        empty_logit = cached_empty[b_name][idx]
        denom       = orig_logit - empty_logit

        # Random ablation
        np.random.seed(RANDOM_SEED + idx + 100000)
        denom_rand = orig_logit - empty_logit
        all_runs   = []
        for _ in range(N_RUNS):
            perm       = np.random.permutation(n_words)
            run_logits = []
            for k in range(n_words + 1):
                text = ablate_text(words, set(perm[:k]), b_name)
                raw  = float(predict_logits(text)[0, predicted_class])
                run_logits.append((raw - empty_logit) / denom_rand)
            all_runs.append(run_logits)
        rand_runs = np.array(all_runs)

        for run in rand_runs:
            curves[b_name]["random"].append(
                np.interp(PCT_GRID, pct_real, run))

        try:
            sv = compute_shap_fixed(Z, W, words, predicted_class,
                                    b_name, empty_logit)

            sign  = np.sign(orig_logit - empty_logit)
            order_tk = np.argsort(sign * sv)[::-1]
            order_bk = np.argsort(sign * sv)

            topk_curve, bottomk_curve = [], []
            for k in range(n_words + 1):
                text_tk = ablate_text(words, set(order_tk[:k]), b_name)
                text_bk = ablate_text(words, set(order_bk[:k]), b_name)
                raw_tk  = float(predict_logits(text_tk)[0, predicted_class])
                raw_bk  = float(predict_logits(text_bk)[0, predicted_class])
                topk_curve.append((raw_tk - empty_logit) / denom)
                bottomk_curve.append((raw_bk - empty_logit) / denom)

            topk_interp    = np.interp(PCT_GRID, pct_real, topk_curve)
            bottomk_interp = np.interp(PCT_GRID, pct_real, bottomk_curve)

            curves[b_name]["topk"].append(topk_interp)
            curves[b_name]["bottomk"].append(bottomk_interp)

            abc_s = float(auc(PCT_GRID / 100, bottomk_interp - topk_interp))
            abc_per_sample[b_name].append(abc_s)

            per_sample_data[b_name].append({
                "index":           idx,
                "text":            SENTIMENT_TEXTS[idx],
                "predicted_class": predicted_class,
                "orig_logit":      orig_logit,
                "empty_logit":     empty_logit,
                "shap_values":     sv.tolist(),
                "topk_norm":       topk_curve,
                "bottomk_norm":    bottomk_curve,
                "abc":             abc_s,
                "n_words":         n_words,
            })

        except Exception as e:
            print(f"  Sample {idx} baseline {b_name} SHAP error: {str(e)[:70]}")
            continue

for b_name in BASELINES:
    n_abc = len(abc_per_sample[b_name])
    print(f"  {b_name.upper()}: ABC = {np.mean(abc_per_sample[b_name]):.4f} +/- "
          f"{np.std(abc_per_sample[b_name]):.4f}  (n={n_abc})")

# aggregate

def agg(arr):
    a = np.array(arr)
    if len(a) == 0:
        return np.zeros(N_GRID), np.zeros(N_GRID)
    return a.mean(0), a.std(0) / np.sqrt(len(a))

agg_curves = {b: {k: agg(curves[b][k])
                  for k in ("random", "topk", "bottomk")}
              for b in BASELINES}

# Save results

with open(SAVE_PATH, "wb") as f:
    pickle.dump({
        "meta": {
            "model":             MODEL_NAME,
            "n_samples":         N_SAMPLES,
            "n_valid":           n_valid,
            "valid_mask_method": VALID_MASK_METHOD,
            "denom_percentile":  DENOM_PERCENTILE,
            "denom_threshold":   float(denom_threshold),
            "logit_iqr_factor":  LOGIT_IQR_FACTOR,
            "shap_mode":         str(SHAP_MODE),
            "n_runs":            N_RUNS,
            "random_seed":       RANDOM_SEED,
            "fixed_coalitions":  True,
            "saved_at":          datetime.datetime.now().isoformat(),
        },
        "pct_grid":        PCT_GRID.tolist(),
        "global_valid":    global_valid.tolist(),
        "min_denoms":      min_denoms.tolist(),
        "abc_per_sample":  abc_per_sample,
        "per_sample_data": per_sample_data,
        "agg_curves": {
            b: {k: {"mean": v[0].tolist(), "se": v[1].tolist()}
                for k, v in agg_curves[b].items()}
            for b in BASELINES
        },
    }, f)
print(f"\nResults saved -> {SAVE_PATH}")

# plot

filter_desc = (
    f"Logit filter: Q1 - {LOGIT_IQR_FACTOR}*IQR, fixed coalitions across baselines"
    if VALID_MASK_METHOD == "logit"
    else f"Denom filter: bottom {DENOM_PERCENTILE}th percentile excluded")

BASELINE_TITLES = {
    "mask":    "Baseline: [MASK] Token",
    "removal": "Baseline: Word Removal",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle(
    f"Ablation Analysis -- DistilBERT  "
    f"(N={n_valid}/{N_SAMPLES} valid, seed={RANDOM_SEED})\n"
    f"Comparison of [MASK] token baseline vs word removal baseline  |  "
    f"{filter_desc}",
    fontsize=11, fontweight='bold')

for ax, b_name in zip(axes, BASELINES):
    rm, rs  = agg_curves[b_name]["random"]
    tm, ts  = agg_curves[b_name]["topk"]
    bm, bs_ = agg_curves[b_name]["bottomk"]

    ax.fill_between(PCT_GRID, tm, bm, alpha=0.12, color='purple',
                    label='Area between curves')
    ax.plot(PCT_GRID, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
    ax.fill_between(PCT_GRID, rm - rs, rm + rs, alpha=0.15, color='blue')
    ax.plot(PCT_GRID, tm, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
    ax.fill_between(PCT_GRID, tm - ts, tm + ts, alpha=0.15, color='red')
    ax.plot(PCT_GRID, bm, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')
    ax.fill_between(PCT_GRID, bm - bs_, bm + bs_, alpha=0.15, color='green')

    ax.set_xlim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated words (%)", fontsize=11)
    ax.set_title(BASELINE_TITLES[b_name], fontsize=11, fontweight='bold')

    n_s     = len(abc_per_sample[b_name])
    abc_std = np.std(abc_per_sample[b_name])  if n_s > 1 else 0.0
    abc_m   = np.mean(abc_per_sample[b_name]) if n_s > 0 else 0.0
    ax.text(0.97, 0.97,
            f"ABC = {abc_m:.3f} +/- {abc_std:.3f}\n(n = {n_s})",
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[0].set_ylabel("Logit as a fraction of original logit", fontsize=10)

all_vals = []
for b_name in BASELINES:
    rm, rs  = agg_curves[b_name]["random"]
    tm, ts  = agg_curves[b_name]["topk"]
    bm, bs_ = agg_curves[b_name]["bottomk"]
    all_vals.extend([
        (rm - rs).min(), (rm + rs).max(),
        (tm - ts).min(), (tm + ts).max(),
        (bm - bs_).min(), (bm + bs_).max(),
    ])
y_lo = min(all_vals) - 0.05
y_hi = max(all_vals) + 0.05
for ax in axes:
    ax.set_ylim(y_lo, y_hi)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig(PLOT_PATH, bbox_inches='tight')
print(f"Plot saved -> {PLOT_PATH}")
plt.show()

# wilcoxon signedrank test

print(f"  SYMMETRY CHECK & WILCOXON SIGNED-RANK TEST")
print(f"  H0: no difference in ABC between [MASK] and word-removal baseline")

v_mask    = np.array(abc_per_sample["mask"])
v_removal = np.array(abc_per_sample["removal"])
min_n     = min(len(v_mask), len(v_removal))
v_mask    = v_mask[:min_n]
v_removal = v_removal[:min_n]
diff      = v_mask - v_removal

print(f"  n = {min_n} paired ABC values\n")

s        = float(skew(diff))
mean_d   = float(np.mean(diff))
median_d = float(np.median(diff))
std_d    = float(np.std(diff))
rel_diff = abs(mean_d - median_d) / (std_d + 1e-10)

print(f"  Symmetry of differences (d = ABC_mask - ABC_removal):")
print(f"    Skewness          : {s:+.3f}  "
      f"({'ok' if abs(s) < 0.5 else 'warning'} threshold |skew| < 0.5)")
print(f"    |mean - median|/sigma : {rel_diff:.3f}  "
      f"({'ok' if rel_diff < 0.2 else 'warning'} threshold < 0.2)")

fig3, axes3 = plt.subplots(1, 2, figsize=(12, 4))
fig3.suptitle(
    "Symmetry Diagnostics for Wilcoxon Signed-Rank Test\n"
    "ABC differences between baselines: ABC([MASK]) - ABC(removal)",
    fontsize=12, fontweight='bold')

ax_h = axes3[0]
ax_h.hist(diff, bins=max(5, min_n // 5), color='steelblue',
          edgecolor='white', alpha=0.85)
ax_h.axvline(0,        color='red',    lw=1.5, ls='--', label='0')
ax_h.axvline(mean_d,   color='orange', lw=1.5, ls='-',
             label=f'mean={mean_d:.3f}')
ax_h.axvline(median_d, color='green',  lw=1.5, ls=':',
             label=f'median={median_d:.3f}')
ax_h.set_xlabel("ABC difference", fontsize=10)
ax_h.set_ylabel("Count", fontsize=10)
ax_h.set_title(f"Histogram of ABC differences\nskew={s:+.3f},  "
               f"|mu-med|/sigma={rel_diff:.3f}", fontsize=10)
ax_h.legend(fontsize=9)
ax_h.grid(True, alpha=0.3)

ax_s = axes3[1]
ax_s.scatter(v_mask, v_removal, alpha=0.6, color='steelblue',
             edgecolors='white', linewidths=0.5, s=50)
lim_lo = min(v_mask.min(), v_removal.min()) - 0.05
lim_hi = max(v_mask.max(), v_removal.max()) + 0.05
ax_s.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1, label='identity line')
ax_s.set_xlabel("ABC  ([MASK] baseline)", fontsize=10)
ax_s.set_ylabel("ABC  (removal baseline)", fontsize=10)
ax_s.set_title("Per-sample ABC: [MASK] vs Removal", fontsize=10)
ax_s.legend(fontsize=9)
ax_s.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DIAG_PATH, bbox_inches='tight')
print(f"  Diagnostics saved -> {DIAG_PATH}\n")
plt.show()

alpha = 0.05
if np.all(diff == 0):
    print("  All differences are zero — test not applicable.")
else:
    stat, p = wilcoxon(v_mask, v_removal, alternative='two-sided')
    sig     = "significant" if p < alpha else "not significant"
    better  = "[MASK]" if mean_d > 0 else "removal"
    print(f"  Wilcoxon Signed-Rank Test (two-sided, alpha={alpha}):")
    print(f"    W={stat:.3f},  p={p:.4f}  -> {sig}")
    print(f"    Higher mean ABC: {better} baseline")
    print(f"    D mean   = {mean_d:+.4f}")
    print(f"    D median = {median_d:+.4f}")
    print()

# Prediction ablation - Resnet18

In [ ]:
%matplotlib inline
import subprocess
import sys

try:
    import shap
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "shap"])

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.linalg import lstsq
from scipy.stats import wilcoxon, skew
from sklearn.metrics import auc
from math import comb
import pickle
import datetime
import warnings
warnings.filterwarnings('ignore')

import tqdm as _tqdm_module
_tqdm_module.tqdm = lambda *a, **kw: (a[0] if a else iter([]))
try:
    _tqdm_module.notebook.tqdm = _tqdm_module.tqdm
    _tqdm_module.auto.tqdm     = _tqdm_module.tqdm
except AttributeError:
    pass

# configuration

RANDOM_SEED = 42069
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

MASKING_STRATEGIES  = ["zero", "blur"]
N_SAMPLES           = 100
N_RUNS              = 20
PATCH_SIZE          = 32
SHAP_MODE           = 1024   # number of coalition samples
BLUR_SIGMA          = 10.0
YAXIS_PERCENTILE_LO = 2
YAXIS_PERCENTILE_HI = 98
RUN_WILCOXON        = True

VALID_MASK_METHOD = "logit"
DENOM_PERCENTILE  = 10
LOGIT_IQR_FACTOR  = 0.5

MODEL_PATH = "resnet18_cifar10_finetuned.pth"
OUTPUT_DIR = r"C:\Users\lukas\OneDrive\Documents\Plots MA test"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Random seed: {RANDOM_SEED}")
print(f"SHAP coalitions: {SHAP_MODE}")
print(f"Filter: {VALID_MASK_METHOD}"
      + (f" (IQR factor: {LOGIT_IQR_FACTOR})" if VALID_MASK_METHOD == "logit"
         else f" (percentile: {DENOM_PERCENTILE})") + "\n")

# data and model

print(f"Loading finetuned ResNet18 from {MODEL_PATH} ...")
model    = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 10)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model    = model.to(device)
model.eval()
print("Model loaded.\n")

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Loading CIFAR10 test set...")
testset    = CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=1, shuffle=False)
print("Data loaded.\n")

# helper functions

def predict_logits(image_batch):
    with torch.no_grad():
        out = model(image_batch.to(device))
    return out.cpu().numpy()

def get_patches(image):
    _, h, w = image.shape
    patches = []
    for y in range(0, h, PATCH_SIZE):
        for x in range(0, w, PATCH_SIZE):
            patches.append((x, min(x + PATCH_SIZE, w),
                             y, min(y + PATCH_SIZE, h)))
    return patches

def mask_patches(image, patch_indices_to_mask, strategy):
    patches = get_patches(image)
    img     = image.clone()
    if strategy == "zero":
        for idx in patch_indices_to_mask:
            x1, x2, y1, y2 = patches[idx]
            img[:, y1:y2, x1:x2] = 0.0
    elif strategy == "blur":
        mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_dn = (img * std_t + mean_t).numpy()
        for idx in patch_indices_to_mask:
            x1, x2, y1, y2 = patches[idx]
            for c in range(3):
                img_dn[c, y1:y2, x1:x2] = gaussian_filter(
                    img_dn[c, y1:y2, x1:x2], sigma=BLUR_SIGMA)
        img = (torch.from_numpy(img_dn).float() - mean_t) / std_t
    return img

def predict_ablated_scalar(image, indices_to_ablate, predicted_class, strategy):
    img_abl = mask_patches(image, indices_to_ablate, strategy)
    return float(predict_logits(img_abl.unsqueeze(0))[0, predicted_class])

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse:
        order = order[::-1]
    return order

def perform_random_ablation(image, predicted_class, strategy, n_patches,
                             empty_scalar, orig_scalar):
    denom    = orig_scalar - empty_scalar
    all_runs = []
    for _ in range(N_RUNS):
        perm = np.random.permutation(n_patches)
        run  = []
        for k in range(n_patches + 1):
            raw = predict_ablated_scalar(
                image, list(perm[:k]), predicted_class, strategy)
            run.append((raw - empty_scalar) / denom)
        all_runs.append(run)
    return np.array(all_runs)

# own kernelshap implementation

def shapley_kernel_weight(z, d):
    s = int(z.sum())
    if s == 0 or s == d:
        return 1e6
    return (d - 1) / (comb(d, s) * s * (d - s))

def sample_coalitions(n_feat, n_samples, seed):
    rng = np.random.RandomState(seed)
    Z   = rng.randint(0, 2, size=(n_samples, n_feat)).astype(float)
    # Fix boundary coalitions for efficiency property
    Z[0] = np.zeros(n_feat)
    Z[1] = np.ones(n_feat)
    return Z

def kernel_shap_wls(Z, W, model_fn, phi_0):
    y     = model_fn(Z) - phi_0
    W_sqrt = np.sqrt(W)
    ZW    = Z * W_sqrt[:, None]
    yW    = y * W_sqrt
    phi, _, _, _ = lstsq(ZW, yW)
    return phi

def compute_shap_values_fixed_coalitions(Z, W, image, predicted_class,
                                          strategy, empty_scalar):
    def model_fn(Z_in):
        preds = []
        for row in Z_in:
            absent = list(np.where(row == 0)[0])
            preds.append(
                predict_ablated_scalar(image, absent, predicted_class, strategy))
        return np.array(preds)

    return kernel_shap_wls(Z, W, model_fn, empty_scalar)

# valid mask

def compute_global_valid_mask(cached_orig, cached_empty, n_actual, strategies,
                               method="denom"):
    min_denoms = np.full(n_actual, np.inf)
    for s in strategies:
        d = np.abs(cached_orig[s] - cached_empty[s])
        min_denoms = np.minimum(min_denoms, d)

    if method == "denom":
        threshold    = np.percentile(min_denoms, DENOM_PERCENTILE)
        global_valid = min_denoms >= threshold
        print(f"  Denom threshold (p{DENOM_PERCENTILE}): {threshold:.4e}  "
              f"| excluded: {int((~global_valid).sum())}")
        return global_valid, threshold, min_denoms

    excluded_by = {i: set() for i in range(n_actual)}
    for s in strategies:
        ref_orig = cached_orig[s]
        q1       = np.percentile(ref_orig, 25)
        q3       = np.percentile(ref_orig, 75)
        iqr      = q3 - q1
        thr      = q1 - LOGIT_IQR_FACTOR * iqr
        flagged  = np.where(ref_orig < thr)[0]
        print(f"  Logit filter [{s:6s}]: "
              f"Q1={q1:.4f}, IQR={iqr:.4f}, "
              f"threshold={thr:.4f} (Q1 - {LOGIT_IQR_FACTOR}*IQR), "
              f"flagged={len(flagged)}")
        for idx in flagged:
            excluded_by[idx].add(s)

    global_valid = np.array([len(excluded_by[i]) == 0 for i in range(n_actual)])
    n_excl = int((~global_valid).sum())
    print(f"  Logit filter total: {n_excl} sample(s) excluded "
          f"(flagged by >=1 strategy)")
    for i in range(n_actual):
        if not global_valid[i]:
            print(f"    sample {i:3d} flagged by: "
                  f"{', '.join(sorted(excluded_by[i]))}")

    return global_valid, 0.0, min_denoms

# first pass: collect orig and empty scalers

N_GRID   = None
PCT_GRID = None

print(f"  RESNET18 PREDICTION ABLATION — CIFAR10")
print("pass 1: collecting orig/empty scalars for all samples")

all_images       = []
all_labels       = []
all_pred_classes = []
cached_orig      = {s: [] for s in MASKING_STRATEGIES}
cached_empty     = {s: [] for s in MASKING_STRATEGIES}

sample_count = 0
for images, labels in testloader:
    if sample_count >= N_SAMPLES:
        break

    image           = images[0]
    label           = int(labels[0].item())
    logits_orig     = predict_logits(images)[0]
    predicted_class = int(np.argmax(logits_orig))
    orig_scalar     = float(logits_orig[predicted_class])

    patches = get_patches(image)
    n_p     = len(patches)
    if N_GRID is None:
        N_GRID   = n_p + 1
        PCT_GRID = np.linspace(0, 100, N_GRID)
        print(f"  Image: {image.shape[1]}x{image.shape[2]} px, "
              f"{n_p} patches ({image.shape[1]//PATCH_SIZE}x"
              f"{image.shape[2]//PATCH_SIZE} grid)\n")

    all_images.append(image)
    all_labels.append(label)
    all_pred_classes.append(predicted_class)

    for s in MASKING_STRATEGIES:
        empty_scalar = predict_ablated_scalar(
            image, list(range(n_p)), predicted_class, s)
        cached_orig[s].append(orig_scalar)
        cached_empty[s].append(empty_scalar)

    sample_count += 1
    if sample_count % 50 == 0:
        print(f"  {sample_count}/{N_SAMPLES} samples scanned", flush=True)

n_actual = sample_count
cached_orig  = {s: np.array(cached_orig[s])  for s in MASKING_STRATEGIES}
cached_empty = {s: np.array(cached_empty[s]) for s in MASKING_STRATEGIES}

# global valid mask

print(f"\nComputing valid mask (method={VALID_MASK_METHOD})...")
global_valid, denom_threshold, min_denoms = compute_global_valid_mask(
    cached_orig, cached_empty, n_actual, MASKING_STRATEGIES,
    method=VALID_MASK_METHOD)
n_valid = int(global_valid.sum())
print(f"Global valid samples: {n_valid}/{n_actual} "
      f"({n_actual - n_valid} excluded)\n")

# pass 2: shap and ablation curves

curves          = {s: {"random": [], "topk": [], "bottomk": []}
                   for s in MASKING_STRATEGIES}
abc_per_sample  = {s: [] for s in MASKING_STRATEGIES}
per_sample_data = {s: [] for s in MASKING_STRATEGIES}

print("Pass 2: computing SHAP values and ablation curves...\n")

for i in range(n_actual):
    if not global_valid[i]:
        continue

    image           = all_images[i]
    predicted_class = all_pred_classes[i]
    label           = all_labels[i]
    patches         = get_patches(image)
    n_p             = len(patches)
    pct_real        = np.linspace(0, 100, n_p + 1)

    Z = sample_coalitions(n_p, SHAP_MODE, seed=RANDOM_SEED + i)
    W = np.array([shapley_kernel_weight(z, n_p) for z in Z])

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  Sample {i+1:3d}/{n_actual}  "
              f"class: {CIFAR10_CLASSES[predicted_class]}", flush=True)

    for strategy in MASKING_STRATEGIES:
        orig_scalar  = cached_orig[strategy][i]
        empty_scalar = cached_empty[strategy][i]
        denom        = orig_scalar - empty_scalar


        np.random.seed(RANDOM_SEED + i + 100000)
        rand_runs = perform_random_ablation(
            image, predicted_class, strategy, n_p, empty_scalar, orig_scalar)
        for run in rand_runs:
            curves[strategy]["random"].append(
                np.interp(PCT_GRID, pct_real, run))

        try:
            # SHAP values using precomputed Z and W
            sv = compute_shap_values_fixed_coalitions(
                Z, W, image, predicted_class, strategy, empty_scalar)

            order_topk    = get_topk_order(sv, orig_scalar, empty_scalar,
                                           inverse=False)
            order_bottomk = get_topk_order(sv, orig_scalar, empty_scalar,
                                           inverse=True)

            topk_curve, bottomk_curve = [], []
            for k in range(n_p + 1):
                topk_curve.append(predict_ablated_scalar(
                    image, list(order_topk[:k]),    predicted_class, strategy))
                bottomk_curve.append(predict_ablated_scalar(
                    image, list(order_bottomk[:k]), predicted_class, strategy))

            topk_norm      = (np.array(topk_curve)    - empty_scalar) / denom
            bottomk_norm   = (np.array(bottomk_curve) - empty_scalar) / denom
            topk_interp    = np.interp(PCT_GRID, pct_real, topk_norm)
            bottomk_interp = np.interp(PCT_GRID, pct_real, bottomk_norm)

            curves[strategy]["topk"].append(topk_interp)
            curves[strategy]["bottomk"].append(bottomk_interp)

            abc_s = float(auc(PCT_GRID / 100, bottomk_interp - topk_interp))
            abc_per_sample[strategy].append(abc_s)

            per_sample_data[strategy].append({
                "sample_index":    i,
                "label":           label,
                "predicted_class": predicted_class,
                "orig_scalar":     orig_scalar,
                "empty_scalar":    empty_scalar,
                "shap_values":     sv.tolist(),
                "topk_norm":       topk_norm.tolist(),
                "bottomk_norm":    bottomk_norm.tolist(),
                "abc":             abc_s,
                "n_patches":       n_p,
            })

        except Exception as e:
            print(f"  SHAP error sample {i} strategy {strategy}: {str(e)[:70]}")
            continue

for strategy in MASKING_STRATEGIES:
    n_abc = len(abc_per_sample[strategy])
    print(f"\n  {strategy.upper()}: ABC_pred = "
          f"{np.mean(abc_per_sample[strategy]):.4f} +/- "
          f"{np.std(abc_per_sample[strategy]):.4f}  (n={n_abc})")

# aggregate and save

def agg(arr):
    a = np.array(arr)
    if len(a) == 0:
        return np.zeros(N_GRID), np.zeros(N_GRID)
    return a.mean(0), a.std(0) / np.sqrt(len(a))

agg_curves = {s: {k: agg(curves[s][k]) for k in ("random", "topk", "bottomk")}
              for s in MASKING_STRATEGIES}

SAVE_PATH = f"{OUTPUT_DIR}/resnet_ablation_cifar10_patch{PATCH_SIZE}.pkl"
with open(SAVE_PATH, "wb") as f:
    pickle.dump({
        "meta": {
            "model":              "resnet18_finetuned_cifar10",
            "dataset":            "cifar10",
            "n_samples":          n_actual,
            "n_valid":            n_valid,
            "valid_mask_method":  VALID_MASK_METHOD,
            "denom_percentile":   DENOM_PERCENTILE,
            "denom_threshold":    float(denom_threshold),
            "logit_iqr_factor":   LOGIT_IQR_FACTOR,
            "n_runs":             N_RUNS,
            "patch_size":         PATCH_SIZE,
            "shap_mode":          SHAP_MODE,
            "random_seed":        RANDOM_SEED,
            "fixed_coalitions":   True,
            "saved_at":           datetime.datetime.now().isoformat(),
        },
        "pct_grid":        PCT_GRID.tolist(),
        "global_valid":    global_valid.tolist(),
        "min_denoms":      min_denoms.tolist(),
        "abc_per_sample":  abc_per_sample,
        "per_sample_data": per_sample_data,
        "agg_curves": {
            s: {k: {"mean": v[0].tolist(), "se": v[1].tolist()}
                for k, v in agg_curves[s].items()}
            for s in MASKING_STRATEGIES
        },
    }, f)
print(f"\nResults saved -> {SAVE_PATH}")

# plot

filter_desc = (
    f"Logit filter: Q1 - {LOGIT_IQR_FACTOR}*IQR per strategy, union"
    if VALID_MASK_METHOD == "logit"
    else f"Denom filter: bottom {DENOM_PERCENTILE}th percentile excluded")

all_mean_vals = []
for s in MASKING_STRATEGIES:
    for k in ("random", "topk", "bottomk"):
        all_mean_vals.extend(agg_curves[s][k][0])
y_lo = min(np.percentile(all_mean_vals, YAXIS_PERCENTILE_LO), -0.05)
y_hi = max(np.percentile(all_mean_vals, YAXIS_PERCENTILE_HI),  1.05)

STRATEGY_TITLES = {
    "zero": "Baseline: Zero Masking",
    "blur": f"Baseline: Gaussian Blur (sigma = {BLUR_SIGMA})",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
fig.suptitle(
    f"Prediction Ablation -- ResNet18 (CIFAR10 finetuned, "
    f"patch = {PATCH_SIZE}x{PATCH_SIZE} px, "
    f"N = {n_valid}/{n_actual} valid, seed = {RANDOM_SEED})\n"
    f"Normalised logit score  |  "
    f"Ablation order: Top-K / Bottom-K by signed SHAP  |  "
    f"{filter_desc}",
    fontsize=11, fontweight='bold')

for ax, s in zip(axes, MASKING_STRATEGIES):
    rm, rs = agg_curves[s]["random"]
    tm, ts = agg_curves[s]["topk"]
    bm, bs = agg_curves[s]["bottomk"]

    ax.fill_between(PCT_GRID, tm, bm, alpha=0.12, color='purple',
                    label='Area between curves')
    ax.plot(PCT_GRID, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
    ax.fill_between(PCT_GRID, rm - rs, rm + rs, alpha=0.15, color='blue')
    ax.plot(PCT_GRID, tm, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
    ax.fill_between(PCT_GRID, tm - ts, tm + ts, alpha=0.15, color='red')
    ax.plot(PCT_GRID, bm, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')
    ax.fill_between(PCT_GRID, bm - bs, bm + bs, alpha=0.15, color='green')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.0, color='grey', lw=0.8, ls=':',  alpha=0.5)
    ax.set_xlim(0, 100)
    ax.set_ylim(y_lo, y_hi)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated patches (%)", fontsize=11)
    ax.set_title(STRATEGY_TITLES[s], fontsize=11, fontweight='bold')

    n_s       = len(abc_per_sample[s])
    abc_m     = np.mean(abc_per_sample[s]) if n_s > 0 else 0.0
    abc_s_val = np.std(abc_per_sample[s])  if n_s > 1 else 0.0
    ax.text(0.97, 0.97,
            f"ABC = {abc_m:.3f} +/- {abc_s_val:.3f}\n(n = {n_s})",
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[0].set_ylabel("Logit as a fraction of original logit", fontsize=10)
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()

plot_fname = f"{OUTPUT_DIR}/resnet_ablation_cifar10_patch{PATCH_SIZE}.pdf"
plt.savefig(plot_fname, bbox_inches='tight')
print(f"Plot saved -> {plot_fname}")
plt.show()

# wilcoxon test

if RUN_WILCOXON:
    print(f"  WILCOXON SIGNED-RANK TEST  (zero vs blur)")

    v_zero = np.array(abc_per_sample["zero"])
    v_blur = np.array(abc_per_sample["blur"])
    n      = min(len(v_zero), len(v_blur))
    v_zero, v_blur = v_zero[:n], v_blur[:n]
    diff   = v_zero - v_blur

    s_val    = float(skew(diff))
    mean_d   = float(np.mean(diff))
    median_d = float(np.median(diff))
    std_d    = float(np.std(diff))
    rel_diff = abs(mean_d - median_d) / (std_d + 1e-10)

    print(f"  n = {n},  skew = {s_val:+.3f},  |mu-med|/sigma = {rel_diff:.3f}")

    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
    fig2.suptitle("Symmetry Diagnostics -- Wilcoxon Signed-Rank Test\n"
                  "ABC_pred differences: ABC(zero) - ABC(blur)",
                  fontsize=12, fontweight='bold')

    ax_h = axes2[0]
    ax_h.hist(diff, bins=max(5, n // 5), color='steelblue',
              edgecolor='white', alpha=0.85)
    ax_h.axvline(0,        color='red',    lw=1.5, ls='--', label='0')
    ax_h.axvline(mean_d,   color='orange', lw=1.5, ls='-',
                 label=f'mean={mean_d:.3f}')
    ax_h.axvline(median_d, color='green',  lw=1.5, ls=':',
                 label=f'median={median_d:.3f}')
    ax_h.set_xlabel("ABC_pred difference (zero - blur)", fontsize=10)
    ax_h.set_ylabel("Count", fontsize=10)
    ax_h.set_title(f"skew={s_val:+.3f},  |mu-med|/sigma={rel_diff:.3f}",
                   fontsize=10)
    ax_h.legend(fontsize=9)
    ax_h.grid(True, alpha=0.3)

    ax_s = axes2[1]
    ax_s.scatter(v_zero, v_blur, alpha=0.6, color='steelblue',
                 edgecolors='white', linewidths=0.5, s=50)
    lim_lo = min(v_zero.min(), v_blur.min()) - 0.05
    lim_hi = max(v_zero.max(), v_blur.max()) + 0.05
    ax_s.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'k--', lw=1,
              label='identity')
    ax_s.set_xlabel("ABC_pred  (zero masking)", fontsize=10)
    ax_s.set_ylabel("ABC_pred  (blur masking)", fontsize=10)
    ax_s.set_title("Per-sample ABC_pred: Zero vs. Blur", fontsize=10)
    ax_s.legend(fontsize=9)
    ax_s.grid(True, alpha=0.3)

    plt.tight_layout()
    diag_fname = (f"{OUTPUT_DIR}/"
                  f"resnet_wilcoxon_diagnostics_patch{PATCH_SIZE}.pdf")
    plt.savefig(diag_fname, bbox_inches='tight')
    print(f"  Diagnostics saved -> {diag_fname}")
    plt.show()

    if not np.all(diff == 0) and len(diff) >= 3:
        stat, p = wilcoxon(v_zero, v_blur, alternative='two-sided')
        sig    = "significant" if p < 0.05 else "not significant"
        better = "zero" if mean_d > 0 else "blur"
        print(f"\n  W = {stat:.1f},  p = {p:.4f}  -> {sig}")
        print(f"  Higher mean ABC_pred: {better} masking")
        print(f"  D mean = {mean_d:+.4f},  D median = {median_d:+.4f}")


print(f"done. All outputs in {OUTPUT_DIR}/")

# Performance ablation - tabular datasets

In [ ]:
matplotlib inline
import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, mean_absolute_error, auc
import torch
import torch.nn as nn
import joblib

#configuration 

DATASET = "california_housing"   # "wine", "california_housing", "diabetes", "adult"

DATASET_CONFIG = {
    "wine":               {"random_state": 6767,  "pkl": r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_wine.pkl",               "model_dir": r"C:\Users\lukas\Documents\Plots MA test", "output_dir": r"C:\Users\lukas\Documents\Plots MA test"},
    "california_housing": {"random_state": 6767,  "pkl": r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_california_housing.pkl", "model_dir": r"C:\Users\lukas\Documents\Plots MA test", "output_dir": r"C:\Users\lukas\Documents\Plots MA test"},
    "diabetes":           {"random_state": 67,    "pkl": r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_diabetes.pkl",           "model_dir": r"C:\Users\lukas\Documents\Plots MA test", "output_dir": r"C:\Users\lukas\Documents\Plots MA test"},
    "adult":              {"random_state": 67,    "pkl": r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_adult.pkl",              "model_dir": r"C:\Users\lukas\Documents\Plots MA test", "output_dir": r"C:\Users\lukas\Documents\Plots MA test"},
}

RBSHAP_SAMPLES = 25
TEST_SIZE      = 0.2
MODELS         = ["random_forest", "xgboost", "mlp"]
BASELINES      = ["mean", "median", "rbshap"]
N_RUNS         = 10

MODEL_LABELS    = {"random_forest": "Random Forest", "xgboost": "XGBoost", "mlp": "MLP"}
BASELINE_LABELS = {"mean": "Mean", "median": "Median", "rbshap": f"RBShap (n={RBSHAP_SAMPLES})"}


cfg          = DATASET_CONFIG[DATASET]
PKL_PATH     = cfg["pkl"]
MODEL_DIR    = cfg["model_dir"]
OUTPUT_DIR   = cfg["output_dir"]
RANDOM_STATE = cfg["random_state"]

# MLP

class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32), nn.ReLU(),
            nn.Linear(32, 32),         nn.ReLU(),
            nn.Linear(32, output_size)
        )
    def forward(self, x): return self.net(x)

#load datasets

def load_data(dataset_name):
    if dataset_name == "wine":
        from sklearn.datasets import load_wine
        d = load_wine()
        return d.data, d.target, list(d.feature_names), True
    elif dataset_name == "california_housing":
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing()
        return d.data, d.target, list(d.feature_names), False
    elif dataset_name == "diabetes":
        from sklearn.datasets import load_diabetes
        d = load_diabetes()
        return d.data, d.target, list(d.feature_names), False
    elif dataset_name == "adult":
        from sklearn.datasets import fetch_openml
        data = fetch_openml('adult', version=2, as_frame=True, parser='auto')
        X = data.data.copy(); y = data.target.copy()
        feature_names = X.columns.tolist()
        for col in X.columns:
            if X[col].dtype == 'object' or X[col].dtype.name == 'category':
                X[col] = LabelEncoder().fit_transform(X[col].astype(str))
        X = np.array(X.astype(np.float64))
        if y.dtype == 'object' or y.dtype.name == 'category':
            y = LabelEncoder().fit_transform(y.astype(str))
        return X, np.array(y), feature_names, True
    raise ValueError(f"Unknown dataset: {dataset_name}")

#load data and split 

print(f"Loading dataset: {DATASET}")
X, y, feature_names, is_classifier = load_data(DATASET)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

n_feat   = X_test_sc.shape[1]
n_actual = len(X_test_sc)
print(f"Task: {'Classification' if is_classifier else 'Regression'}, features: {n_feat}\n")

# load pkl

with open(PKL_PATH, "rb") as f:
    saved = pickle.load(f)

global_valid  = np.array(saved["global_valid"])
valid_indices = np.where(global_valid)[0]
n_valid       = len(valid_indices)
print(f"Valid samples: {n_valid}/{n_actual}\n")

y_valid = y_test[valid_indices]

# model wrapper classes

def to_logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1.0 - p))

class MLPWrapper:
    def __init__(self, params):
        self.params        = params
        self.model         = None
        self.is_classifier = False
        self.device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def predict_raw(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()

    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.argmax(dim=1).cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()

class ModelWrapper:
    def __init__(self, model_type, params):
        self.model_type    = model_type
        self.params        = params
        self.model         = None
        self.is_classifier = False

    def predict_scalar(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        if self.is_classifier:
            if self.model_type == "mlp":
                logits       = self.model.predict_raw(X)
                pred_classes = logits.argmax(axis=1)
                return logits[np.arange(len(logits)), pred_classes]
            else:
                proba        = self.model.predict_proba(X)
                pred_classes = proba.argmax(axis=1)
                return to_logit(proba[np.arange(len(proba)), pred_classes])
        else:
            if self.model_type == "mlp":
                return self.model.predict_raw(X).flatten()
            return self.model.predict(X).flatten()

    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1, -1)
        if self.model_type == "mlp":
            return self.model.predict(X)
        return self.model.model.predict(X)

# models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(RANDOM_STATE)

rf  = joblib.load(f"{MODEL_DIR}/model_{DATASET}_random_forest.pkl")
xgb = joblib.load(f"{MODEL_DIR}/model_{DATASET}_xgboost.pkl")

n_out     = len(np.unique(y_train)) if is_classifier else 1
mlp_model = MLP(n_feat, n_out).to(device)
mlp_model.load_state_dict(torch.load(
    f"{MODEL_DIR}/model_{DATASET}_mlp.pt", map_location=device))
mlp_model.eval()

trained_models = {"random_forest": rf, "xgboost": xgb, "mlp": mlp_model}

# predict helpers

def predict_sklearn(wrapper, X_in):
    """Predict class or value from ModelWrapper."""
    if is_classifier:
        return wrapper.model.predict(X_in)
    else:
        return wrapper.model.predict(X_in)

def predict_mlp(X_in):
    with torch.no_grad():
        out = mlp_model(torch.FloatTensor(X_in).to(device))
    if is_classifier:
        return out.argmax(dim=1).cpu().numpy()
    else:
        result = out.squeeze().cpu().numpy()
        return np.atleast_1d(result)

def predict_ablated(m_name, x, features_to_remove, baseline):
    is_rbshap = isinstance(baseline, np.ndarray) and baseline.ndim == 2
    if is_rbshap:
        X_in = np.tile(x, (len(baseline), 1))
        if len(features_to_remove) > 0:
            X_in[:, list(features_to_remove)] = baseline[:, list(features_to_remove)]
        if m_name == "mlp":
            with torch.no_grad():
                out = mlp_model(torch.FloatTensor(X_in).to(device))
            if is_classifier:
                return int(out.mean(0).argmax())
            else:
                return float(out.squeeze().cpu().numpy().mean())
        else:
            if is_classifier:
                return int(trained_models[m_name].model.predict_proba(X_in).mean(0).argmax())
            else:
                return float(trained_models[m_name].model.predict(X_in).mean())
    else:
        x_abl = x.copy()
        if len(features_to_remove) > 0:
            x_abl[list(features_to_remove)] = baseline[list(features_to_remove)]
        X_in = x_abl.reshape(1, -1)
        if m_name == "mlp":
            if is_classifier:
                return int(predict_mlp(X_in)[0])
            else:
                return float(predict_mlp(X_in)[0])
        else:
            if is_classifier:
                return int(predict_sklearn(trained_models[m_name], X_in)[0])
            else:
                return float(predict_sklearn(trained_models[m_name], X_in)[0])

def predict_batch(m_name, baseline):
    preds = [predict_ablated(m_name, X_test_sc[idx], [], baseline)
             for idx in valid_indices]
    return np.array(preds)

# metric

def score(y_true, y_pred, baseline_score=None):
    if is_classifier:
        s = f1_score(y_true, y_pred, average="macro", zero_division=0)
        return s / baseline_score if baseline_score else s
    else:
        s = mean_absolute_error(y_true, y_pred)
        # for regression: lower MAE is better, so we invert normalization
        # normalized = baseline_MAE / current_MAE (higher = better)
        return baseline_score / s if baseline_score else s

# baselines

baselines_vec = {
    "mean":   np.mean(X_train_sc, axis=0),
    "median": np.median(X_train_sc, axis=0),
    "rbshap": X_train_sc[np.random.choice(
                  len(X_train_sc), size=RBSHAP_SAMPLES, replace=False)],
}

# signed shap order

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse: order = order[::-1]
    return order

#performance ablation

percentages = np.linspace(0, 100, n_feat + 1)
pss         = saved["per_sample_store"]

abc_perf    = {b: {m: None for m in MODELS} for b in BASELINES}
perf_curves = {b: {m: {} for m in MODELS} for b in BASELINES}

for b_name in BASELINES:
    baseline = baselines_vec[b_name]

    for m_name in MODELS:
        print(f"  {b_name.upper()} | {m_name.upper()}")
        entries = {e["index"]: e for e in pss[b_name][m_name]}

        # baseline score at k=0
        preds_0      = predict_batch(m_name, baseline)
        baseline_scr = score(y_valid, preds_0)
        print(f"    {'F1' if is_classifier else 'MAE'} at k=0: {baseline_scr:.4f}")

        topk_curve    = []
        bottomk_curve = []

        for k in range(n_feat + 1):
            preds_tk = []
            preds_bk = []

            for idx in valid_indices:
                entry        = entries[idx]
                shap_vals    = np.array(entry["shap_values"])
                orig_scalar  = entry["orig_scalar"]
                empty_scalar = entry["empty_scalar"]

                order_tk = get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False)
                order_bk = get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=True)

                preds_tk.append(predict_ablated(m_name, X_test_sc[idx],
                                                list(order_tk[:k]), baseline))
                preds_bk.append(predict_ablated(m_name, X_test_sc[idx],
                                                list(order_bk[:k]), baseline))

            preds_tk = np.array(preds_tk)
            preds_bk = np.array(preds_bk)

            topk_curve.append(score(y_valid, preds_tk, baseline_scr))
            bottomk_curve.append(score(y_valid, preds_bk, baseline_scr))

            if k % 3 == 0:
                print(f"    k={k}/{n_feat}", flush=True)

        topk_arr    = np.array(topk_curve)
        bottomk_arr = np.array(bottomk_curve)

        # random ablation
        random_curves = []
        for _ in range(N_RUNS):
            perm       = np.random.permutation(n_feat)
            rand_curve = []
            for k in range(n_feat + 1):
                preds_r = np.array([
                    predict_ablated(m_name, X_test_sc[idx], list(perm[:k]), baseline)
                    for idx in valid_indices])
                rand_curve.append(score(y_valid, preds_r, baseline_scr))
            random_curves.append(rand_curve)

        random_arr = np.array(random_curves)

        perf_curves[b_name][m_name] = {
            "topk":    topk_arr,
            "bottomk": bottomk_arr,
            "random":  random_arr,
        }

        abc = float(auc(percentages / 100, bottomk_arr - topk_arr))
        abc_perf[b_name][m_name] = abc
        print(f"    ABC_perf = {abc:.4f}\n")

# 3x3 plot

metric_label = "Normalised Macro-F1" if is_classifier else "Normalised MAE (baseline / current)"

fig, axes = plt.subplots(3, 3, figsize=(18, 14), sharex=True, sharey=True)
fig.suptitle(
    f"Performance Ablation — {DATASET.replace('_', ' ').title()}  "
    f"(N={n_valid}/{n_actual} valid, seed={RANDOM_STATE})\n"
    f"{metric_label}  |  Ablation order: Top-K / Bottom-K by signed SHAP",
    fontsize=11, fontweight='bold', y=1.02)

for row, b_name in enumerate(BASELINES):
    for col, m_name in enumerate(MODELS):
        ax = axes[row][col]
        d  = perf_curves[b_name][m_name]
        tk = d["topk"]
        bk = d["bottomk"]
        rm = d["random"].mean(axis=0)
        rs = d["random"].std(axis=0) / np.sqrt(N_RUNS)

        ax.fill_between(percentages, tk, bk, alpha=0.12, color='purple',
                        label='Area between curves')
        ax.plot(percentages, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
        ax.fill_between(percentages, rm - rs, rm + rs, alpha=0.15, color='blue')
        ax.plot(percentages, tk, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
        ax.plot(percentages, bk, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')

        ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
        ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
        ax.set_xlim(0, 100)
        ax.grid(True, alpha=0.3)

        abc = abc_perf[b_name][m_name]
        ax.text(0.97, 0.97, f"ABC={abc:.3f}",
                transform=ax.transAxes, fontsize=8, ha='right', va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

        if row == 0: ax.set_title(MODEL_LABELS[m_name], fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel(metric_label, fontsize=8)
            ax.text(-0.28, 0.5, BASELINE_LABELS[b_name], transform=ax.transAxes,
                    fontsize=11, fontweight='bold', va='center', ha='center', rotation=90)
        if col == 2: ax.tick_params(axis='y', labelright=True)
        if row == 2: ax.set_xlabel("Ablated features (%)", fontsize=9)

handles, labels_leg = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=11, bbox_to_anchor=(0.5, -0.03))
plt.tight_layout()

plot_fname = f"{OUTPUT_DIR}/performance_ablation_{DATASET}.pdf"
plt.savefig(plot_fname, bbox_inches='tight')
print(f"Plot saved -> {plot_fname}")
plt.show()

# abc summary

print(f"\n{'='*55}")
print(f"  ABC_perf SUMMARY — {DATASET.upper()}")
print(f"{'='*55}")
for b_name in BASELINES:
    print(f"\n  {b_name}")
    for m_name in MODELS:
        print(f"    {m_name:15s}: {abc_perf[b_name][m_name]:.4f}")

# Performance ablation - DistilBERT

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all')
import pickle
import numpy as np
import torch
from sklearn.metrics import f1_score, auc
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# configuration

PKL_PATH   = "distilbert_sst2_results.pkl"
OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA"

RANDOM_SEED = 67676767
N_RUNS      = 10
MODEL_NAME  = "distilbert-base-uncased-finetuned-sst-2-english"
BASELINES   = ["mask", "removal"]

BASELINE_TITLES = {
    "mask":    "Baseline: [MASK] Token",
    "removal": "Baseline: Word Removal",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(RANDOM_SEED)

#load model

print("Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
model     = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded.\n")

# load pkl

with open(PKL_PATH, 'rb') as f:
    saved = pickle.load(f)

global_valid = np.array(saved['global_valid'])
pct_grid     = np.array(saved['pct_grid'])
pss          = saved['per_sample_data']
n_valid      = int(global_valid.sum())
print(f"Valid samples: {n_valid}")

# helper functions

def predict_logits(texts):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(texts, return_tensors="pt", padding=True,
                       truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.logits.cpu().numpy()

def ablate_text(words, indices_to_remove, baseline_type):
    if baseline_type == "mask":
        return " ".join(
            "[MASK]" if i in indices_to_remove else w
            for i, w in enumerate(words))
    else:
        kept = [w for i, w in enumerate(words) if i not in indices_to_remove]
        return " ".join(kept) if kept else "[PAD]"

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse: order = order[::-1]
    return order

# true labels are predicted classes at k=0

# use mask baseline entries as reference (same predicted_class for both)
true_labels = {}
for entry in pss['mask']:
    true_labels[entry['index']] = entry['predicted_class']

valid_indices = [i for i, v in enumerate(global_valid) if v]
y_true = np.array([true_labels[i] for i in valid_indices])
print(f"True labels (predicted class at k=0): {dict(zip(*np.unique(y_true, return_counts=True)))}\n")

# sentences

SENTIMENT_TEXTS = [
    "I remember when this came out a lot of kids were nuts about it",
    "This is a great movie for the true romantics and sports lovers alike",
    "This is strictly a review of the pilot episode as it appears on DVD",
    "I was quite impressed with this movie as a child of eight or nine",
    "When Family Guy first premiered I was not in a discriminating mood",
    "Tom Clancy uses Alesandr Nevsky in his book Red Storm Rising",
    "I won't spend a lot of time nor energy on this comment",
    "This was such a terrible film almost a comedy sketch of a noir film",
    "Well maybe I'm just having a bad run with Hindi movies lately",
    "The Duke is a very silly film--a dog becoming a duke",
    "More exciting than the Wesley Snipes film and with better characters too",
    "This movie was Jerry Bruckheimer's idea to sell some records",
    "Karl Jr and his dad are now running an army on a remote island",
    "The film has no connection with the real life in Bosnia in those days",
    "No doubt Frank Sinatra was a talented actor as well as a talented singer",
    "Yesterday my Spanish Catalan wife and myself saw this emotional lesson in history",
    "The 3rd and in my view the best of the Blackadder series",
    "With some films it is really hard to tell for whom they were made",
    "Not only is this film entertaining with excellent comedic acting but also interesting politically",
    "This was Eisenstein's first completed project in over ten years",
    "Many of these other viewers complain that the story line has already been attempted",
    "Oh boy it's another comet-hitting-the-earth film",
    "First of all DO NOT call this a remake of the 63 film",
    "Great little short film that aired a while ago on SBS here in Aus",
    "If you really loved GWTW you will find quite disappointing the story",
    "Probably New Zealands worst Movie ever made The Jokes They are not funny",
    "This film is about the worst I have seen in a very long time",
    "This is by far the worst movie I have ever seen in the cinema",
    "If anybody really wants to understand Hitler read WWI history not WWII history",
    "This movie is more Lupin then most especially coming from Funimation",
]

MAX_WORDS = max(len(t.split()) for t in SENTIMENT_TEXTS)
N_GRID    = MAX_WORDS + 1
PCT_GRID  = np.linspace(0, 100, N_GRID)

#performance ablation

perf_curves = {b: {"topk": None, "bottomk": None, "random": None}
               for b in BASELINES}
abc_perf    = {b: None for b in BASELINES}

for b_name in BASELINES:
    print(f"\n{'─'*55}")
    print(f"  Baseline: {b_name.upper()}")
    print(f"{'─'*55}")

    entries    = {e['index']: e for e in pss[b_name]}
    topk_curve = []
    botk_curve = []

    for k in range(N_GRID):
        preds_tk = []
        preds_bk = []
        for idx in valid_indices:
            entry      = entries[idx]
            words      = SENTIMENT_TEXTS[idx].split()
            shap_vals  = np.array(entry['shap_values'])
            orig       = entry['orig_logit']
            empty      = entry['empty_logit']
            n_words    = len(words)
            pct_real   = np.linspace(0, 100, n_words + 1)

            # map k (grid index) to actual word index
            k_actual = int(np.round(k / (N_GRID - 1) * n_words))
            k_actual = min(k_actual, n_words)

            order_tk = get_topk_order(shap_vals, orig, empty, inverse=False)
            order_bk = get_topk_order(shap_vals, orig, empty, inverse=True)

            text_tk = ablate_text(words, set(order_tk[:k_actual]), b_name)
            text_bk = ablate_text(words, set(order_bk[:k_actual]), b_name)

            preds_tk.append(int(np.argmax(predict_logits(text_tk)[0])))
            preds_bk.append(int(np.argmax(predict_logits(text_bk)[0])))

        topk_curve.append(f1_score(y_true, preds_tk, average='macro', zero_division=0))
        botk_curve.append(f1_score(y_true, preds_bk, average='macro', zero_division=0))

        if k % 3 == 0:
            print(f"  k={k}/{N_GRID-1}", flush=True)

    f1_0 = topk_curve[0]
    topk_norm = np.array(topk_curve) / f1_0
    botk_norm = np.array(botk_curve) / f1_0

    # random ablation
    rand_runs = []
    for _ in range(N_RUNS):
        rand_curve = []
        for k in range(N_GRID):
            preds_r = []
            for idx in valid_indices:
                words    = SENTIMENT_TEXTS[idx].split()
                n_words  = len(words)
                perm     = np.random.permutation(n_words)
                k_actual = int(np.round(k / (N_GRID - 1) * n_words))
                k_actual = min(k_actual, n_words)
                text_r   = ablate_text(words, set(perm[:k_actual]), b_name)
                preds_r.append(int(np.argmax(predict_logits(text_r)[0])))
            rand_curve.append(f1_score(y_true, preds_r, average='macro', zero_division=0) / f1_0)
        rand_runs.append(rand_curve)

    rand_arr = np.array(rand_runs)
    rand_mean = rand_arr.mean(0)
    rand_se   = rand_arr.std(0) / np.sqrt(N_RUNS)

    perf_curves[b_name] = {
        "topk":      topk_norm,
        "bottomk":   botk_norm,
        "random":    rand_mean,
        "random_se": rand_se,
    }

    abc = float(auc(PCT_GRID / 100, botk_norm - topk_norm))
    abc_perf[b_name] = abc
    print(f"  ABC_perf = {abc:.4f}")

# plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle(
    f"Performance Ablation -- DistilBERT  (N={n_valid}/30 valid, seed={RANDOM_SEED})\n"
    f"Normalised Macro-F1  |  Ablation order: Top-K / Bottom-K by signed SHAP",
    fontsize=11, fontweight='bold')

for ax, b_name in zip(axes, BASELINES):
    d  = perf_curves[b_name]
    rm = d['random']
    rs = d['random_se']
    tk = d['topk']
    bk = d['bottomk']

    ax.fill_between(PCT_GRID, tk, bk, alpha=0.12, color='purple',
                    label='Area between curves')
    ax.plot(PCT_GRID, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
    ax.fill_between(PCT_GRID, rm - rs, rm + rs, alpha=0.15, color='blue')
    ax.plot(PCT_GRID, tk, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
    ax.plot(PCT_GRID, bk, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
    ax.set_xlim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated words (%)", fontsize=10)
    ax.set_title(BASELINE_TITLES[b_name], fontsize=11, fontweight='bold')
    ax.text(0.97, 0.97, f"ABC = {abc_perf[b_name]:.3f}",
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[0].set_ylabel("Normalised Macro-F1", fontsize=10)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()

out_path = f"{OUTPUT_DIR}/performance_ablation_distilbert.pdf"
plt.savefig(out_path, bbox_inches='tight')
print(f"\nPlot saved -> {out_path}")
plt.show()

# Performance ablation - Resnet18

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all')
import pickle
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import CIFAR10
from scipy.ndimage import gaussian_filter
from sklearn.metrics import f1_score, auc

# configuration

PKL_PATH   = r"C:\Users\lukas\Documents\Plots MA\resnet_ablation_cifar10_patch32.pkl"
MODEL_PATH = r"C:\Users\lukas\Documents\Masterarbeit\resnet18_cifar10_finetuned.pth"
OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA"

PATCH_SIZE  = 32
BLUR_SIGMA  = 10.0
RANDOM_SEED = 42069
N_RUNS      = 10

STRATEGIES = ["zero", "blur"]
STRATEGY_TITLES = {
    "zero": "Baseline: Zero Masking",
    "blur": f"Baseline: Gaussian Blur (sigma={BLUR_SIGMA})",
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(RANDOM_SEED)

# load model

print("Loading ResNet18...")
model    = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 10)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model    = model.to(device)
model.eval()
print("Model loaded.\n")

#load data and pkl

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
testset = CIFAR10(root='./data', train=False, download=True, transform=transform)

with open(PKL_PATH, 'rb') as f:
    saved = pickle.load(f)

global_valid  = np.array(saved['global_valid'])
valid_indices = [i for i, v in enumerate(global_valid) if v]
n_valid       = len(valid_indices)
pct_grid      = np.array(saved['pct_grid'])
pss           = saved['per_sample_data']
n_patches     = pss['zero'][0]['n_patches']
N_GRID        = n_patches + 1
PCT_GRID      = np.linspace(0, 100, N_GRID)

print(f"Valid samples: {n_valid}, patches: {n_patches}")

# true labels from testset
y_true = np.array([testset[i][1] for i in valid_indices])
print(f"Label distribution: {dict(zip(*np.unique(y_true, return_counts=True)))}\n")

# helper functions

def get_patches(image):
    _, h, w = image.shape
    patches = []
    for y in range(0, h, PATCH_SIZE):
        for x in range(0, w, PATCH_SIZE):
            patches.append((x, min(x + PATCH_SIZE, w),
                             y, min(y + PATCH_SIZE, h)))
    return patches

def mask_patches(image, patch_indices, strategy):
    patches = get_patches(image)
    img     = image.clone()
    if strategy == "zero":
        for idx in patch_indices:
            x1, x2, y1, y2 = patches[idx]
            img[:, y1:y2, x1:x2] = 0.0
    elif strategy == "blur":
        mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_dn = (img * std_t + mean_t).numpy()
        for idx in patch_indices:
            x1, x2, y1, y2 = patches[idx]
            for c in range(3):
                img_dn[c, y1:y2, x1:x2] = gaussian_filter(
                    img_dn[c, y1:y2, x1:x2], sigma=BLUR_SIGMA)
        img = (torch.from_numpy(img_dn).float() - mean_t) / std_t
    return img

def predict_class(img_tensor):
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
    return int(out.argmax(dim=1).cpu())

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse: order = order[::-1]
    return order

# performance ablation

perf_curves = {s: {} for s in STRATEGIES}
abc_perf    = {s: None for s in STRATEGIES}

for strategy in STRATEGIES:
    print(f"\n{'─'*55}")
    print(f"  Strategy: {strategy.upper()}")
    print(f"{'─'*55}")

    entries = {e['sample_index']: e for e in pss[strategy]}

    # F1 at k=0
    preds_0 = []
    for idx in valid_indices:
        img = testset[idx][0]
        preds_0.append(predict_class(img))
    f1_0 = f1_score(y_true, preds_0, average='macro', zero_division=0)
    print(f"  F1 at k=0: {f1_0:.4f}")

    topk_curve = []
    botk_curve = []

    for k in range(N_GRID):
        preds_tk = []
        preds_bk = []
        for idx in valid_indices:
            entry     = entries[idx]
            image     = testset[idx][0]
            shap_vals = np.array(entry['shap_values'])
            orig      = entry['orig_scalar']
            empty     = entry['empty_scalar']
            order_tk  = get_topk_order(shap_vals, orig, empty, inverse=False)
            order_bk  = get_topk_order(shap_vals, orig, empty, inverse=True)
            img_tk    = mask_patches(image, list(order_tk[:k]), strategy)
            img_bk    = mask_patches(image, list(order_bk[:k]), strategy)
            preds_tk.append(predict_class(img_tk))
            preds_bk.append(predict_class(img_bk))

        topk_curve.append(f1_score(y_true, preds_tk, average='macro', zero_division=0) / f1_0)
        botk_curve.append(f1_score(y_true, preds_bk, average='macro', zero_division=0) / f1_0)

        if k % 5 == 0:
            print(f"  k={k}/{n_patches}", flush=True)

    topk_norm = np.array(topk_curve)
    botk_norm = np.array(botk_curve)

    # random ablation
    rand_runs = []
    for run in range(N_RUNS):
        rand_curve = []
        perm = np.random.permutation(n_patches)
        for k in range(N_GRID):
            preds_r = []
            for idx in valid_indices:
                image  = testset[idx][0]
                img_r  = mask_patches(image, list(perm[:k]), strategy)
                preds_r.append(predict_class(img_r))
            rand_curve.append(
                f1_score(y_true, preds_r, average='macro', zero_division=0) / f1_0)
        rand_runs.append(rand_curve)
        print(f"  Random run {run+1}/{N_RUNS}", flush=True)

    rand_arr  = np.array(rand_runs)
    rand_mean = rand_arr.mean(0)
    rand_se   = rand_arr.std(0) / np.sqrt(N_RUNS)

    perf_curves[strategy] = {
        "topk":      topk_norm,
        "bottomk":   botk_norm,
        "random":    rand_mean,
        "random_se": rand_se,
    }

    abc = float(auc(PCT_GRID / 100, botk_norm - topk_norm))
    abc_perf[strategy] = abc
    print(f"  ABC_perf = {abc:.4f}")

# plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle(
    f"Performance Ablation -- ResNet18 CIFAR10  (N={n_valid}/100 valid, seed={RANDOM_SEED})\n"
    f"Normalised Macro-F1  |  Ablation order: Top-K / Bottom-K by signed SHAP",
    fontsize=11, fontweight='bold')

for ax, strategy in zip(axes, STRATEGIES):
    d  = perf_curves[strategy]
    rm = d['random']
    rs = d['random_se']
    tk = d['topk']
    bk = d['bottomk']

    ax.fill_between(PCT_GRID, tk, bk, alpha=0.12, color='purple',
                    label='Area between curves')
    ax.plot(PCT_GRID, rm, 'b-o', lw=2, ms=3, label='Random Ablation')
    ax.fill_between(PCT_GRID, rm - rs, rm + rs, alpha=0.15, color='blue')
    ax.plot(PCT_GRID, tk, 'r-s', lw=2, ms=3, label='Top-K (SHAP)')
    ax.plot(PCT_GRID, bk, 'g-^', lw=2, ms=3, label='Bottom-K (SHAP)')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
    ax.set_xlim(0, 100)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated patches (%)", fontsize=10)
    ax.set_title(STRATEGY_TITLES[strategy], fontsize=11, fontweight='bold')
    ax.text(0.97, 0.97, f"ABC = {abc_perf[strategy]:.3f}",
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[0].set_ylabel("Normalised Macro-F1", fontsize=10)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()

out_path = f"{OUTPUT_DIR}/performance_ablation_resnet.pdf"
plt.savefig(out_path, bbox_inches='tight')
print(f"\nPlot saved -> {out_path}")
plt.show()

# Disentanglement analysis - Wine

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all')
import pickle
import numpy as np
import joblib
import torch
import torch.nn as nn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, auc

#configuration

PKL_PATH   = r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_wine.pkl"
MODEL_DIR  = r"C:\Users\lukas\Documents\Plots MA test"
OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA test"

RANDOM_STATE   = 6767
TEST_SIZE      = 0.2
RBSHAP_SAMPLES = 25
MODELS         = ["random_forest", "xgboost", "mlp"]
BASELINES      = ["mean", "median", "rbshap"]

# mlp

class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32), nn.ReLU(),
            nn.Linear(32, 32),         nn.ReLU(),
            nn.Linear(32, output_size)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load data and models

d = load_wine()
X, y = d.data, d.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
n_feat     = X_test_sc.shape[1]

def to_logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1.0 - p))

class MLPWrapper:
    def __init__(self, params):
        self.params = params; self.model = None; self.is_classifier = False
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    def predict_raw(self, X):
        if X.ndim == 1: X = X.reshape(1,-1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()
    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1,-1)
        with torch.no_grad():
            out = self.model(torch.FloatTensor(X).to(self.device))
        return out.argmax(dim=1).cpu().numpy() if self.is_classifier else out.squeeze().cpu().numpy().flatten()

class ModelWrapper:
    def __init__(self, model_type, params):
        self.model_type = model_type; self.params = params
        self.model = None; self.is_classifier = False
    def predict_scalar(self, X):
        if X.ndim == 1: X = X.reshape(1,-1)
        if self.is_classifier:
            if self.model_type == "mlp":
                logits = self.model.predict_raw(X)
                pred_classes = logits.argmax(axis=1)
                return logits[np.arange(len(logits)), pred_classes]
            else:
                proba = self.model.predict_proba(X)
                pred_classes = proba.argmax(axis=1)
                return to_logit(proba[np.arange(len(proba)), pred_classes])
        else:
            if self.model_type == "mlp": return self.model.predict_raw(X).flatten()
            return self.model.predict(X).flatten()
    def predict(self, X):
        if X.ndim == 1: X = X.reshape(1,-1)
        if self.model_type == "mlp": return self.model.predict(X)
        return self.model.model.predict(X)

rf  = joblib.load(f"{MODEL_DIR}/model_wine_random_forest.pkl")
xgb = joblib.load(f"{MODEL_DIR}/model_wine_xgboost.pkl")


mlp_model = MLP(n_feat, 3).to(device)
mlp_model.load_state_dict(torch.load(
    f"{MODEL_DIR}/model_wine_mlp.pt", map_location=device))
mlp_model.eval()

trained_models = {"random_forest": rf, "xgboost": xgb, "mlp": mlp_model}

def predict_class(m_name, X_in):
    if m_name == "mlp":
        with torch.no_grad():
            out = trained_models[m_name](torch.FloatTensor(X_in).to(device))
        return out.argmax(dim=1).cpu().numpy()
    return trained_models[m_name].model.predict(X_in)

#load pkl

with open(PKL_PATH, 'rb') as f:
    saved = pickle.load(f)

global_valid  = np.array(saved['global_valid'])
valid_indices = np.where(global_valid)[0]
n_valid       = len(valid_indices)
pss           = saved['per_sample_store']
percentages   = np.array(saved['percentages'])

y_valid = y_test[valid_indices]
print(f"Valid samples: {n_valid}")

# baselines

np.random.seed(RANDOM_STATE)
baselines_vec = {
    "mean":   np.mean(X_train_sc, axis=0),
    "median": np.median(X_train_sc, axis=0),
    "rbshap": X_train_sc[np.random.choice(
                  len(X_train_sc), size=RBSHAP_SAMPLES, replace=False)],
}

def ablate_sample_predict(m_name, x, features_to_remove, baseline):
    """Predict class for x with features_to_remove replaced by baseline."""
    is_rbshap = isinstance(baseline, np.ndarray) and baseline.ndim == 2
    if is_rbshap:
        X_in = np.tile(x, (len(baseline), 1))
        if len(features_to_remove) > 0:
            X_in[:, list(features_to_remove)] = baseline[:, list(features_to_remove)]
        if m_name == "mlp":
            with torch.no_grad():
                out = trained_models[m_name](torch.FloatTensor(X_in).to(device))
            avg = out.mean(dim=0).cpu().numpy()
            return int(avg.argmax())
        else:
            proba = trained_models[m_name].model.predict_proba(X_in)
            return int(proba.mean(axis=0).argmax())
    else:
        x_abl = x.copy()
        if len(features_to_remove) > 0:
            x_abl[list(features_to_remove)] = baseline[list(features_to_remove)]
        return int(predict_class(m_name, x_abl.reshape(1, -1))[0])

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse: order = order[::-1]
    return order

# compute curves

def compute_perf_curve(m_name, order_baseline, replace_baseline):
    order_bl_vec   = baselines_vec[replace_baseline]  # for actual ablation
    replace_bl_vec = baselines_vec[replace_baseline]

    # F1 at k=0
    X_valid   = X_test_sc[valid_indices]
    preds_0   = np.array([ablate_sample_predict(m_name, x, [], replace_bl_vec)
                          for x in X_valid])
    f1_0      = f1_score(y_valid, preds_0, average="macro", zero_division=0)

    curve = []
    for k in range(n_feat + 1):
        preds = []
        for idx in valid_indices:
            entry      = next(e for e in pss[order_baseline][m_name]
                              if e['index'] == idx)
            shap_vals  = np.array(entry['shap_values'])
            orig       = entry['orig_scalar']
            empty      = entry['empty_scalar']
            order      = get_topk_order(shap_vals, orig, empty, inverse=False)
            x          = X_test_sc[idx]
            preds.append(ablate_sample_predict(
                m_name, x, list(order[:k]), replace_bl_vec))
        f1_k = f1_score(y_valid, preds, average="macro", zero_division=0)
        curve.append(f1_k / f1_0 if f1_0 > 0 else 0.0)
        if k % 3 == 0:
            print(f"    k={k}/{n_feat}", flush=True)

    return np.array(curve)

results = {}
for m_name in MODELS:
    print(f"\n{'='*55}")
    print(f"  {m_name.upper()}")
    print(f"{'='*55}")
    results[m_name] = {}

    print("  A: Mean order + Mean replacement...")
    results[m_name]['A_mean_mean']     = compute_perf_curve(m_name, 'mean',   'mean')
    print("  B: RBShap order + Mean replacement...")
    results[m_name]['B_rbshap_mean']   = compute_perf_curve(m_name, 'rbshap', 'mean')
    print("  C: Mean order + RBShap replacement...")
    results[m_name]['C_mean_rbshap']   = compute_perf_curve(m_name, 'mean',   'rbshap')
    print("  D: RBShap order + RBShap replacement...")
    results[m_name]['D_rbshap_rbshap'] = compute_perf_curve(m_name, 'rbshap', 'rbshap')

#plot

MODEL_LABELS = {"random_forest": "Random Forest", "xgboost": "XGBoost", "mlp": "MLP"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    "Wine: Disentangling SHAP order effect vs baseline replacement effect\n"
    "Top-K performance ablation (normalised Macro-F1)",
    fontsize=11, fontweight='bold')

for col, m_name in enumerate(MODELS):
    ax = axes[col]
    r  = results[m_name]

    ax.plot(percentages, r['A_mean_mean'],     color='royalblue',  lw=2, ms=3, marker='o',
            label='A: Mean order + Mean repl.')
    ax.plot(percentages, r['B_rbshap_mean'],   color='darkorange',  lw=2, ms=3, marker='s',
            label='B: RBShap order + Mean repl.')
    ax.plot(percentages, r['C_mean_rbshap'],   color='green',  lw=2, ms=3, marker='^',
            label='C: Mean order + RBShap repl.')
    ax.plot(percentages, r['D_rbshap_rbshap'], color='red', lw=2, ms=3, marker='D',
            label='D: RBShap order + RBShap repl.')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
    ax.set_xlim(0, 100); ax.set_ylim(-0.05, 1.25)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated features (%)", fontsize=10)
    ax.set_title(MODEL_LABELS[m_name], fontsize=11, fontweight='bold')

    # ABC values
    abc_A = float(auc(percentages/100, 1 - r['A_mean_mean']))
    abc_B = float(auc(percentages/100, 1 - r['B_rbshap_mean']))
    abc_C = float(auc(percentages/100, 1 - r['C_mean_rbshap']))
    abc_D = float(auc(percentages/100, 1 - r['D_rbshap_rbshap']))
    ax.text(0.97, 0.97,
            f"A: {abc_A:.3f}\nB: {abc_B:.3f}\nC: {abc_C:.3f}\nD: {abc_D:.3f}",
            transform=ax.transAxes, fontsize=7, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    if col == 0:
        ax.set_ylabel("Normalised Macro-F1", fontsize=10)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()

out_path = f"{OUTPUT_DIR}/wine_disentangle_order_vs_replacement.pdf"
plt.savefig(out_path, bbox_inches='tight')
print(f"\nPlot saved -> {out_path}")
plt.show()

# Disentanglement analysis - California Housing

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all')
import pickle
import numpy as np
import joblib
import torch
import torch.nn as nn
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, auc

# configuration

PKL_PATH   = r"C:\Users\lukas\Documents\Plots MA test\prediction_ablation_california_housing.pkl"
MODEL_DIR  = r"C:\Users\lukas\Documents\Plots MA test"
OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA test"

RANDOM_STATE   = 6767
TEST_SIZE      = 0.2
RBSHAP_SAMPLES = 25
MODELS         = ["random_forest", "xgboost", "mlp"]
BASELINES      = ["mean", "median", "rbshap"]

# mlp

class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32), nn.ReLU(),
            nn.Linear(32, 32),         nn.ReLU(),
            nn.Linear(32, output_size)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load data and models

d = fetch_california_housing()
X, y = d.data, d.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
n_feat     = X_test_sc.shape[1]

rf  = joblib.load(f"{MODEL_DIR}/model_california_housing_random_forest.pkl")
xgb = joblib.load(f"{MODEL_DIR}/model_california_housing_xgboost.pkl")

mlp_model = MLP(n_feat, 1).to(device)
mlp_model.load_state_dict(torch.load(
    f"{MODEL_DIR}/model_california_housing_mlp.pt", map_location=device))
mlp_model.eval()

trained_models = {"random_forest": rf, "xgboost": xgb, "mlp": mlp_model}

def predict_regression(m_name, X_in):
    if m_name == "mlp":
        with torch.no_grad():
            out = trained_models[m_name](torch.FloatTensor(X_in).to(device))
        return out.squeeze(-1).cpu().numpy()
    model = trained_models[m_name]
    if hasattr(model, 'model') and hasattr(model.model, 'predict'):
        return model.model.predict(X_in)
    return model.predict(X_in)

# load pkl

with open(PKL_PATH, 'rb') as f:
    saved = pickle.load(f)

global_valid  = np.array(saved['global_valid'])
valid_indices = np.where(global_valid)[0]
n_valid       = len(valid_indices)
pss           = saved['per_sample_store']
percentages   = np.array(saved['percentages'])

y_valid = y_test[valid_indices]
print(f"Valid samples: {n_valid}")

# baselines

np.random.seed(RANDOM_STATE)
baselines_vec = {
    "mean":   np.mean(X_train_sc, axis=0),
    "median": np.median(X_train_sc, axis=0),
    "rbshap": X_train_sc[np.random.choice(
                  len(X_train_sc), size=RBSHAP_SAMPLES, replace=False)],
}

def ablate_sample_predict(m_name, x, features_to_remove, baseline):
    """Predict regression output for x with features_to_remove replaced by baseline."""
    is_rbshap = isinstance(baseline, np.ndarray) and baseline.ndim == 2
    if is_rbshap:
        X_in = np.tile(x, (len(baseline), 1))
        if len(features_to_remove) > 0:
            X_in[:, list(features_to_remove)] = baseline[:, list(features_to_remove)]
        return float(predict_regression(m_name, X_in).mean())
    else:
        x_abl = x.copy()
        if len(features_to_remove) > 0:
            x_abl[list(features_to_remove)] = baseline[list(features_to_remove)]
        return float(predict_regression(m_name, x_abl.reshape(1, -1))[0])

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse: order = order[::-1]
    return order

# compute curves

def compute_perf_curve(m_name, order_baseline, replace_baseline):
    replace_bl_vec = baselines_vec[replace_baseline]
    X_valid        = X_test_sc[valid_indices]

    preds_0 = np.array([ablate_sample_predict(m_name, x, [], replace_bl_vec)
                        for x in X_valid])
    mae_0   = mean_absolute_error(y_valid, preds_0)
    print(f"    MAE at k=0: {mae_0:.4f}")

    curve = []
    for k in range(n_feat + 1):
        preds = []
        for idx in valid_indices:
            entry     = next(e for e in pss[order_baseline][m_name]
                             if e['index'] == idx)
            shap_vals = np.array(entry['shap_values'])
            orig      = entry['orig_scalar']
            empty     = entry['empty_scalar']
            # BOTTOM-K: inverse=True
            order     = get_topk_order(shap_vals, orig, empty, inverse=True)
            x         = X_test_sc[idx]
            preds.append(ablate_sample_predict(
                m_name, x, list(order[:k]), replace_bl_vec))

        mae_k = mean_absolute_error(y_valid, preds)
        curve.append(mae_0 / mae_k if mae_k > 0 else 0.0)

        if k % 2 == 0:
            print(f"    k={k}/{n_feat}  MAE={mae_k:.4f}  norm={curve[-1]:.4f}", flush=True)

    return np.array(curve)

results = {}
for m_name in MODELS:
    print(f"\n{'='*55}")
    print(f"  {m_name.upper()}")
    print(f"{'='*55}")
    results[m_name] = {}

    print("  A: Mean order + Mean replacement...")
    results[m_name]['A_mean_mean']       = compute_perf_curve(m_name, 'mean',   'mean')
    print("  B: Median order + Mean replacement...")
    results[m_name]['B_median_mean']     = compute_perf_curve(m_name, 'median', 'mean')
    print("  C: Mean order + Median replacement...")
    results[m_name]['C_mean_median']     = compute_perf_curve(m_name, 'mean',   'median')
    print("  D: Median order + Median replacement...")
    results[m_name]['D_median_median']   = compute_perf_curve(m_name, 'median', 'median')

# plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle(
    "California Housing: Disentangling SHAP order effect vs baseline replacement effect\n"
    "Bottom-K performance ablation (normalised MAE)",
    fontsize=11, fontweight='bold')

for col, m_name in enumerate(MODELS):
    ax = axes[col]
    r  = results[m_name]

    ax.plot(percentages, r['A_mean_mean'],     color='royalblue',  lw=2, ms=3, marker='o',
            label='A: Mean order + Mean repl.')
    ax.plot(percentages, r['B_median_mean'],   color='darkorange',  lw=2, ms=3, marker='s',
            label='B: Median order + Mean repl.')
    ax.plot(percentages, r['C_mean_median'],   color='green',  lw=2, ms=3, marker='^',
            label='C: Mean order + Median repl.')
    ax.plot(percentages, r['D_median_median'], color='red', lw=2, ms=3, marker='D',
            label='D: Median order + Median repl.')

    ax.axhline(1.0, color='grey', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0.0, color='grey', lw=0.8, ls=':', alpha=0.5)
    ax.set_xlim(0, 100); ax.set_ylim(-0.05, 1.25)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Ablated features (%)", fontsize=10)
    ax.set_title(MODEL_LABELS[m_name], fontsize=11, fontweight='bold')

    abc_A = float(auc(percentages/100, 1 - r['A_mean_mean']))
    abc_B = float(auc(percentages/100, 1 - r['B_median_mean']))
    abc_C = float(auc(percentages/100, 1 - r['C_mean_median']))
    abc_D = float(auc(percentages/100, 1 - r['D_median_median']))
    ax.text(0.97, 0.97,
            f"A: {abc_A:.3f}\nB: {abc_B:.3f}\nC: {abc_C:.3f}\nD: {abc_D:.3f}",
            transform=ax.transAxes, fontsize=7, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    if col == 0:
        ax.set_ylabel("Normalised MAE", fontsize=10)

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()

out_path = f"{OUTPUT_DIR}/california_housing_disentangle_bottomk_mean_vs_median.pdf"
plt.savefig(out_path, bbox_inches='tight')
print(f"\nPlot saved -> {out_path}")
plt.show()

# Resnet18 - illustration of zero and blur masking

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all')
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import CIFAR10
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import pickle

plt.close('all')

#configuration

PKL_PATH   = r"C:\Users\lukas\Documents\Plots MA\resnet_ablation_cifar10_patch32.pkl"
MODEL_PATH = r"C:\Users\lukas\Documents\Masterarbeit\resnet18_cifar10_finetuned.pth"
OUTPUT_DIR = r"C:\Users\lukas\Documents\Plots MA"

PATCH_SIZE  = 32
BLUR_SIGMA  = 10.0
K_SHOW      = 5
K_SHOW_80   = 40  
RANDOM_SEED = 42069

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load model

model    = models.resnet18(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 10)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model    = model.to(device)
model.eval()

# load pkl

with open(PKL_PATH, 'rb') as f:
    saved = pickle.load(f)

zero_entries = saved['per_sample_data']['zero']
zero_by_idx  = {e['sample_index']: e for e in zero_entries}
common       = sorted(zero_by_idx.keys())

# fixed sample indices
chosen = [14, 33, 73]
print(f"Chosen samples: {chosen}")
print(f"Classes: {[CIFAR10_CLASSES[zero_by_idx[i]['predicted_class']] for i in chosen]}")

#load images

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
testset = CIFAR10(root='./data', train=False, download=True, transform=transform)

# helper functions

def get_patches(image):
    _, h, w = image.shape
    patches = []
    for y in range(0, h, PATCH_SIZE):
        for x in range(0, w, PATCH_SIZE):
            patches.append((x, min(x + PATCH_SIZE, w),
                             y, min(y + PATCH_SIZE, h)))
    return patches

def apply_zero(image, pidx):
    img = image.clone()
    ps  = get_patches(img)
    for p in pidx:
        x1, x2, y1, y2 = ps[p]
        img[:, y1:y2, x1:x2] = 0.0
    return img

def apply_blur(image, pidx):
    mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img    = image.clone()
    img_dn = (img * std_t + mean_t).numpy()
    ps     = get_patches(image)
    for p in pidx:
        x1, x2, y1, y2 = ps[p]
        for c in range(3):
            img_dn[c, y1:y2, x1:x2] = gaussian_filter(
                img_dn[c, y1:y2, x1:x2], sigma=BLUR_SIGMA)
    return (torch.from_numpy(img_dn).float() - mean_t) / std_t

def get_logit(img_tensor, pred_class):
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
    return float(out[0, pred_class].cpu())

def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = img_tensor.numpy().transpose(1, 2, 0)
    return np.clip(std * img + mean, 0, 1)

def draw_patch_boxes(ax, image, pidx, color):
    ps = get_patches(image)
    for p in pidx:
        x1, x2, y1, y2 = ps[p]
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1,
                              linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)

def get_topk_order(shap_vals, orig_scalar, empty_scalar, inverse=False):
    sigma = np.sign(orig_scalar - empty_scalar)
    order = np.argsort(sigma * shap_vals)
    if not inverse:
        order = order[::-1]
    return order

# compute results

results = []
for idx in chosen:
    image      = testset[idx][0]
    entry      = zero_by_idx[idx]
    pred_class = entry['predicted_class']
    label      = entry['label']

    sv           = np.array(entry['shap_values'])
    orig_scalar  = entry['orig_scalar']
    empty_scalar = entry['empty_scalar']

    order_topk    = list(get_topk_order(sv, orig_scalar, empty_scalar, inverse=False))
    order_bottomk = list(get_topk_order(sv, orig_scalar, empty_scalar, inverse=True))

    topk_patches    = order_topk[:K_SHOW]
    bottomk_patches    = order_bottomk[:K_SHOW]
    bottomk_patches_80 = order_bottomk[:K_SHOW_80]

    logit_orig         = get_logit(image, pred_class)
    logit_topk_zero    = get_logit(apply_zero(image, topk_patches),       pred_class)
    logit_topk_blur    = get_logit(apply_blur(image, topk_patches),       pred_class)
    logit_botk_zero    = get_logit(apply_zero(image, bottomk_patches),    pred_class)
    logit_botk_blur    = get_logit(apply_blur(image, bottomk_patches),    pred_class)
    logit_botk80_zero  = get_logit(apply_zero(image, bottomk_patches_80), pred_class)
    logit_botk80_blur  = get_logit(apply_blur(image, bottomk_patches_80), pred_class)

    print(f"  Sample {idx:3d} ({CIFAR10_CLASSES[pred_class]:12s}): "
          f"orig={logit_orig:.2f}  "
          f"topk_zero={logit_topk_zero:.2f}  topk_blur={logit_topk_blur:.2f}  "
          f"botk_zero={logit_botk_zero:.2f}  botk_blur={logit_botk_blur:.2f}")

    results.append({
        'idx':              idx,
        'image':            image,
        'img_topk_zero':    apply_zero(image, topk_patches),
        'img_topk_blur':    apply_blur(image, topk_patches),
        'img_botk_zero':    apply_zero(image, bottomk_patches),
        'img_botk_blur':    apply_blur(image, bottomk_patches),
        'img_botk80_zero':  apply_zero(image, bottomk_patches_80),
        'img_botk80_blur':  apply_blur(image, bottomk_patches_80),
        'pred_class':       pred_class,
        'label':            label,
        'logit_orig':       logit_orig,
        'logit_topk_zero':  logit_topk_zero,
        'logit_topk_blur':  logit_topk_blur,
        'logit_botk_zero':  logit_botk_zero,
        'logit_botk_blur':  logit_botk_blur,
        'logit_botk80_zero': logit_botk80_zero,
        'logit_botk80_blur': logit_botk80_blur,
        'topk_patches':     topk_patches,
        'bottomk_patches':  bottomk_patches,
        'bottomk_patches_80': bottomk_patches_80,
    })

# plot

fig, axes = plt.subplots(3, 7, figsize=(22, 13))

col_titles = [
    "Original",
    f"Top-{K_SHOW} Zero masking",
    f"Top-{K_SHOW} Gaussian Blur",
    f"Bottom-{K_SHOW} Zero masking",
    f"Bottom-{K_SHOW} Gaussian Blur",
    f"Bottom-{K_SHOW_80} Zero masking\n(~80% ablated)",
    f"Bottom-{K_SHOW_80} Gaussian Blur\n(~80% ablated)",
]
col_colors = [None, 'red', 'red', 'green', 'green', 'darkgreen', 'darkgreen']

for col, (title, color) in enumerate(zip(col_titles, col_colors)):
    axes[0][col].set_title(title, fontsize=9, fontweight='bold',
                           color=color if color else 'black', pad=10)

for row, res in enumerate(results):
    drop_topk_zero  = res['logit_orig'] - res['logit_topk_zero']
    drop_topk_blur  = res['logit_orig'] - res['logit_topk_blur']
    drop_botk_zero  = res['logit_orig'] - res['logit_botk_zero']
    drop_botk_blur  = res['logit_orig'] - res['logit_botk_blur']
    drop_botk80_zero = res['logit_orig'] - res['logit_botk80_zero']
    drop_botk80_blur = res['logit_orig'] - res['logit_botk80_blur']

    imgs = [
        res['image'],
        res['img_topk_zero'],
        res['img_topk_blur'],
        res['img_botk_zero'],
        res['img_botk_blur'],
        res['img_botk80_zero'],
        res['img_botk80_blur'],
    ]
    patch_sets = [
        res['topk_patches'],
        res['topk_patches'],
        res['topk_patches'],
        res['bottomk_patches'],
        res['bottomk_patches'],
        res['bottomk_patches_80'],
        res['bottomk_patches_80'],
    ]
    box_colors = ['orange', 'red', 'red', 'green', 'green', 'darkgreen', 'darkgreen']
    subtitles = [
        f"{CIFAR10_CLASSES[res['pred_class']]}\nlogit: {res['logit_orig']:.2f}",
        f"logit: {res['logit_topk_zero']:.2f}  drop: {drop_topk_zero:+.2f}",
        f"logit: {res['logit_topk_blur']:.2f}  drop: {drop_topk_blur:+.2f}",
        f"logit: {res['logit_botk_zero']:.2f}  drop: {drop_botk_zero:+.2f}",
        f"logit: {res['logit_botk_blur']:.2f}  drop: {drop_botk_blur:+.2f}",
        f"logit: {res['logit_botk80_zero']:.2f}  drop: {drop_botk80_zero:+.2f}",
        f"logit: {res['logit_botk80_blur']:.2f}  drop: {drop_botk80_blur:+.2f}",
    ]
    subtitle_colors = ['black', 'red', 'red', 'green', 'green', 'darkgreen', 'darkgreen']

    for col, (img_t, ps, bc, sub, sc) in enumerate(
            zip(imgs, patch_sets, box_colors, subtitles, subtitle_colors)):
        ax = axes[row][col]
        ax.imshow(denormalize(img_t))
        draw_patch_boxes(ax, img_t, ps, color=bc)
        ax.axis('off')
        ax.text(0.5, -0.10, sub, transform=ax.transAxes,
                fontsize=14, ha='center', va='top', color=sc,
                fontweight='bold')

# add vertical separator lines between top-k and bottom-k groups
for row in range(3):
    for col in [0, 2]:
        pos = axes[row][col].get_position()

# column group labels
fig.text(0.25, 0.98, f"Top-{K_SHOW} ablation (most important patches)",
         ha='center', fontsize=11, fontweight='bold', color='red')
fig.text(0.57, 0.98, f"Bottom-{K_SHOW} ablation (least important patches)",
         ha='center', fontsize=11, fontweight='bold', color='green')
fig.text(0.86, 0.98, f"Bottom-{K_SHOW_80} ablation (~80%)",
         ha='center', fontsize=11, fontweight='bold', color='darkgreen')

plt.tight_layout(rect=[0, 0, 1, 0.97], h_pad=6.0)
out_path = f"{OUTPUT_DIR}/resnet_topk_bottomk_comparison.pdf"
plt.savefig(out_path, bbox_inches='tight')
print(f"\nPlot saved -> {out_path}")
plt.show()